In [ ]:
# In[1]:

import pandas as pd
import pytz

sh = pytz.timezone('Asia/Shanghai')

# Load CSVs (reuse variables across operations)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to compute compact time range (as strings in UTC+8)
def time_range_from_series(ts_series):
    if ts_series.empty:
        return {"min": None, "max": None}
    mn = pd.to_datetime(ts_series.min(), unit='s', utc=True).tz_convert(sh).strftime('%Y-%m-%d %H:%M:%S %Z')
    mx = pd.to_datetime(ts_series.max(), unit='s', utc=True).tz_convert(sh).strftime('%Y-%m-%d %H:%M:%S %Z')
    return {"min": mn, "max": mx}

# Metric summary
metric_count = int(df_metric.shape[0])
metric_time = time_range_from_series(df_metric['timestamp'])
metric_cmdbs = sorted(df_metric['cmdb_id'].dropna().unique().tolist())
metric_unique_cmdb_count = len(metric_cmdbs)
metric_pairs = (df_metric.groupby(['cmdb_id', 'kpi_name'])
                .size()
                .reset_index(name='count')
                .sort_values('count', ascending=False)
               )
metric_top20 = metric_pairs.head(20).to_dict(orient='records')

# Trace summary
trace_count = int(df_trace.shape[0])
trace_time = time_range_from_series(df_trace['timestamp'])
trace_cmdbs = sorted(df_trace['cmdb_id'].dropna().unique().tolist())
trace_unique_cmdb_count = len(trace_cmdbs)
trace_pairs = (df_trace.groupby(['cmdb_id', 'trace_name'])
               .size()
               .reset_index(name='count')
               .sort_values('count', ascending=False)
              )
trace_top20 = trace_pairs.head(20).to_dict(orient='records')

# Log summary
log_count = int(df_log.shape[0])
log_time = time_range_from_series(df_log['timestamp'])
log_cmdbs = sorted(df_log['cmdb_id'].dropna().unique().tolist())
log_unique_cmdb_count = len(log_cmdbs)
log_pairs = (df_log.groupby(['cmdb_id', 'log_name'])
             .size()
             .reset_index(name='count')
             .sort_values('count', ascending=False)
            )
log_top20 = log_pairs.head(20).to_dict(orient='records')

# Error logs summary
error_count = int(df_error.shape[0])
error_time = time_range_from_series(df_error['timestamp'])
error_cmdbs = sorted(df_error['cmdb_id'].dropna().unique().tolist())
error_unique_cmdb_count = len(error_cmdbs)

# Determine if messages are long -> switch to per-cmdb aggregation with samples
max_msg_len = int(df_error['message'].str.len().fillna(0).max()) if not df_error['message'].empty else 0
long_message_threshold = 200
if max_msg_len > long_message_threshold:
    # aggregated counts per cmdb_id and up to 5 distinct sample messages
    grp = df_error.groupby('cmdb_id').agg(
        total_rows=('message','size'),
        unique_messages=('message','nunique'),
        earliest_ts=('timestamp','min'),
        latest_ts=('timestamp','max')
    ).reset_index()
    # convert timestamps
    grp['earliest'] = grp['earliest_ts'].apply(lambda x: pd.to_datetime(x, unit='s', utc=True).tz_convert(sh).strftime('%Y-%m-%d %H:%M:%S %Z'))
    grp['latest'] = grp['latest_ts'].apply(lambda x: pd.to_datetime(x, unit='s', utc=True).tz_convert(sh).strftime('%Y-%m-%d %H:%M:%S %Z'))
    # sample messages
    samples = {}
    for cm in grp['cmdb_id'].tolist():
        msgs = df_error[df_error['cmdb_id']==cm]['message'].drop_duplicates().head(5).tolist()
        samples[cm] = msgs
    error_agg = grp[['cmdb_id','total_rows','unique_messages','earliest','latest']].to_dict(orient='records')
    error_detail = {"mode":"per_cmdb_aggregated","max_message_length":max_msg_len,"per_cmdb_summary":error_agg,"sample_messages_per_cmdb":samples}
else:
    # return top 20 (cmdb_id, message) pairs with counts
    err_pairs = (df_error.groupby(['cmdb_id','message'])
                 .size()
                 .reset_index(name='count')
                 .sort_values('count', ascending=False)
                )
    error_top20 = err_pairs.head(20).to_dict(orient='records')
    error_detail = {"mode":"top_pairs","max_message_length":max_msg_len,"top_pairs":error_top20}

# Assemble final JSON-friendly summary (compact)
summary = {
    "metric": {
        "row_count": metric_count,
        "time_range": metric_time,
        "unique_cmdb_ids_count": metric_unique_cmdb_count,
        "unique_cmdb_ids": metric_cmdbs,
        "top_20_cmdb_kpi_pairs": metric_top20
    },
    "trace": {
        "row_count": trace_count,
        "time_range": trace_time,
        "unique_cmdb_ids_count": trace_unique_cmdb_count,
        "unique_cmdb_ids": trace_cmdbs,
        "top_20_cmdb_trace_pairs": trace_top20
    },
    "log": {
        "row_count": log_count,
        "time_range": log_time,
        "unique_cmdb_ids_count": log_unique_cmdb_count,
        "unique_cmdb_ids": log_cmdbs,
        "top_20_cmdb_log_pairs": log_top20
    },
    "error_logs": {
        "row_count": error_count,
        "time_range": error_time,
        "unique_cmdb_ids_count": error_unique_cmdb_count,
        "unique_cmdb_ids": error_cmdbs,
        "detail": error_detail
    }
}

summary

```
Out[1]:
```
```python
# Summarize execution results in plain English (compact)
s = summary  # reuse previous variable from kernel

metric = s['metric']
trace = s['trace']
log = s['log']
error = s['error_logs']

# pick a few top pairs for brief mention
def top_pairs_text(pairs, n=3):
    if not pairs:
        return "none"
    items = pairs[:n]
    return "; ".join([f"{p['cmdb_id']} | {p[list(p.keys())[1]]} ({p['count']})" for p in items])

conclusion = (
    "Summary of telemetry files:\n\n"
    f"1) metric.csv: {metric['row_count']} rows from {metric['time_range']['min']} to {metric['time_range']['max']} (UTC+8). "
    f"{metric['unique_cmdb_ids_count']} unique cmdb_ids. Top metric (cmdb_id,kpi) pairs include: {top_pairs_text(metric['top_20_cmdb_kpi_pairs'])}.\n\n"
    f"2) trace.csv: {trace['row_count']} rows from {trace['time_range']['min']} to {trace['time_range']['max']} (UTC+8). "
    f"{trace['unique_cmdb_ids_count']} unique cmdb_ids. Top trace pairs include: {top_pairs_text(trace['top_20_cmdb_trace_pairs'])}.\n\n"
    f"3) log.csv: {log['row_count']} rows from {log['time_range']['min']} to {log['time_range']['max']} (UTC+8). "
    f"{log['unique_cmdb_ids_count']} unique cmdb_ids. Top log pairs include: {top_pairs_text(log['top_20_cmdb_log_pairs'])}.\n\n"
    f"4) error_logs.csv: {error['row_count']} rows (no error log records found).\n\n"
    "Notes: Metric data is the largest source. Many frequent metric entries come from Tomcat instances (Tomcat03/Tomcat04) and IG01. "
    "Trace data is concentrated on trace.self.* metrics (IG/MG and some docker/Tomcat). Log data shows apache01 and Tomcat instances with the highest row/error counts. "
    "There are no raw error_log messages to inspect."
)

conclusion
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric': {'row_count': 28281, 'time_range': {'min': '2021-03-25 02:30:00 CST', 'max': '2021-03-25 03:00:00 CST'}, 'unique_cmdb_ids_count': 26, 'unique_cmdb_ids': ['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'ServiceTest1', 'ServiceTest10', 'ServiceTest11', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4', 'ServiceTest5', 'ServiceTest6', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02', 'dockerA2'], 'top_20_cmdb_kpi_pairs': [{'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SESSIONRejectedSessions', 'count': 31}, {'cmdb_id': 'IG01', 'kpi_name': 'JVM-Threads_7778_JVM_ThreadCount_Threads', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SESSIONRejectedSessions', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SESSIONActiveCounter', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SESSIONKeepaliveCounter', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SESSIONRejectedSessions', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadCountThreadInfo', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadsBusyThreadInfo', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Threads_7441-"http-nio-8003"_MaxThreadsThreadInfo', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCounter', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--UOCP_SESSIONRejectedSessions', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--_SESSIONActiveCounter', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--_SESSIONKeepaliveCounter', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--_SESSIONRejectedSessions', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SESSIONActiveCounter', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SESSIONKeepaliveCounter', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_ErrorCountRequestInfo', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo', 'count': 31}]}, 'trace': {'row_count': 4179, 'time_range': {'min': '2021-03-25 02:30:00 CST', 'max': '2021-03-25 02:59:00 CST'}, 'unique_cmdb_ids_count': 12, 'unique_cmdb_ids': ['IG01', 'IG02', 'MG01', 'MG02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'], 'top_20_cmdb_trace_pairs': [{'cmdb_id': 'IG01', 'trace_name': 'trace.self.duration_mean', 'count': 27}, {'cmdb_id': 'IG01', 'trace_name': 'trace.self.duration_p95', 'count': 27}, {'cmdb_id': 'IG01', 'trace_name': 'trace.self.row_count', 'count': 27}, {'cmdb_id': 'IG02', 'trace_name': 'trace.self.duration_p95', 'count': 27}, {'cmdb_id': 'IG02', 'trace_name': 'trace.self.duration_mean', 'count': 27}, {'cmdb_id': 'IG02', 'trace_name': 'trace.self.row_count', 'count': 27}, {'cmdb_id': 'MG02', 'trace_name': 'trace.self.duration_p95', 'count': 27}, {'cmdb_id': 'MG02', 'trace_name': 'trace.self.duration_mean', 'count': 27}, {'cmdb_id': 'MG02', 'trace_name': 'trace.self.row_count', 'count': 27}, {'cmdb_id': 'MG01', 'trace_name': 'trace.self.duration_p95', 'count': 27}, {'cmdb_id': 'MG01', 'trace_name': 'trace.self.row_count', 'count': 27}, {'cmdb_id': 'MG01', 'trace_name': 'trace.self.duration_mean', 'count': 27}, {'cmdb_id': 'dockerB1', 'trace_name': 'trace.self.row_count', 'count': 26}, {'cmdb_id': 'Tomcat01', 'trace_name': 'trace.self.row_count', 'count': 26}, {'cmdb_id': 'dockerB1', 'trace_name': 'trace.self.duration_p95', 'count': 26}, {'cmdb_id': 'dockerB1', 'trace_name': 'trace.self.duration_mean', 'count': 26}, {'cmdb_id': 'Tomcat01', 'trace_name': 'trace.self.duration_mean', 'count': 26}, {'cmdb_id': 'Tomcat01', 'trace_name': 'trace.self.duration_p95', 'count': 26}, {'cmdb_id': 'MG02', 'trace_name': 'trace.to_dockerB1.duration_p95', 'count': 25}, {'cmdb_id': 'MG02', 'trace_name': 'trace.to_dockerB1.row_count', 'count': 25}]}, 'log': {'row_count': 294, 'time_range': {'min': '2021-03-25 02:30:00 CST', 'max': '2021-03-25 02:59:00 CST'}, 'unique_cmdb_ids_count': 6, 'unique_cmdb_ids': ['Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02'], 'top_20_cmdb_log_pairs': [{'cmdb_id': 'apache01', 'log_name': 'log.row_count', 'count': 27}, {'cmdb_id': 'apache01', 'log_name': 'log.error_count', 'count': 27}, {'cmdb_id': 'Tomcat01', 'log_name': 'log.error_count', 'count': 26}, {'cmdb_id': 'Tomcat01', 'log_name': 'log.row_count', 'count': 26}, {'cmdb_id': 'Tomcat02', 'log_name': 'log.error_count', 'count': 24}, {'cmdb_id': 'Tomcat02', 'log_name': 'log.row_count', 'count': 24}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.error_count', 'count': 23}, {'cmdb_id': 'apache02', 'log_name': 'log.row_count', 'count': 23}, {'cmdb_id': 'apache02', 'log_name': 'log.error_count', 'count': 23}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.row_count', 'count': 23}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.error_count', 'count': 22}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.row_count', 'count': 22}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.gc_pause_max', 'count': 2}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.gc_pause_max', 'count': 1}, {'cmdb_id': 'Tomcat01', 'log_name': 'log.gc_pause_max', 'count': 1}]}, 'error_logs': {'row_count': 0, 'time_range': {'min': None, 'max': None}, 'unique_cmdb_ids_count': 0, 'unique_cmdb_ids': [], 'detail': {'mode': 'top_pairs', 'max_message_length': 0, 'top_pairs': []}}}```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse df_metric if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# timezone (UTC+8)
sh = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series (before any filtering)
# ensure value is numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
global_p95 = (
    df_metric
    .dropna(subset=['value'])
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Convert timestamps to Asia/Shanghai timezone-aware datetimes and filter to incident window
df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_metric['ts_sh'] = df_metric['ts_utc'].dt.tz_convert(sh)

# define window in Asia/Shanghai tz
window_start_sh = pd.Timestamp('2021-03-25 02:30:00').tz_localize(sh)
window_end_sh = pd.Timestamp('2021-03-25 03:00:00').tz_localize(sh)

df_window = df_metric[(df_metric['ts_sh'] >= window_start_sh) & (df_metric['ts_sh'] <= window_end_sh)].copy()

# 3) For each (cmdb_id,kpi_name) present in the filtered window compute required fields
# merge global_p95 into windowed rows
df_win = df_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# mark anomalies where value >= global_p95 (note: global_p95 is computed from full series)
df_win['is_anomaly'] = df_win['value'] >= df_win['global_p95']

# aggregate per pair
agg_counts = (
    df_win
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        points_in_window=('value', 'size'),
        anomaly_count=('is_anomaly', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest anomaly timestamp per pair (None if no anomalies)
earliest_anom = (
    df_win[df_win['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['ts_sh']
    .min()
    .rename(columns={'ts_sh': 'earliest_anomaly_ts'})
)

# combine aggregations with global_p95
result_df = (
    agg_counts
    .merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')
)

# format earliest_anomaly_ts as string in UTC+8 or None
result_df['earliest_anomaly_timestamp'] = result_df['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
result_df.loc[result_df['earliest_anomaly_ts'].isna(), 'earliest_anomaly_timestamp'] = None

# select and order columns as requested
result_df = result_df[[
    'cmdb_id',
    'kpi_name',
    'global_p95',
    'points_in_window',
    'anomaly_count',
    'earliest_anomaly_timestamp',
    'max_value_in_window'
]]

# 4) Sort by anomaly_count desc, then points_in_window desc. Return top 20 pairs and total count with anomaly_count>0
result_df_sorted = result_df.sort_values(['anomaly_count', 'points_in_window'], ascending=[False, False]).reset_index(drop=True)
top20_df = result_df_sorted.head(20).copy()

total_pairs_with_anomalies = int((result_df['anomaly_count'] > 0).sum())

# Prepare JSON-friendly output (compact). Convert numeric types to native Python types where appropriate.
top20_records = top20_df.to_dict(orient='records')

result = {
    "window_start": window_start_sh.strftime('%Y-%m-%d %H:%M:%S %Z'),
    "window_end": window_end_sh.strftime('%Y-%m-%d %H:%M:%S %Z'),
    "total_pairs_in_window": int(result_df.shape[0]),
    "total_pairs_with_anomalies": total_pairs_with_anomalies,
    "top_20_pairs": top20_records
}

result

```
Out[2]:
```
Summary (plain English):

- Window analyzed: 2021-03-25 02:30:00 to 2021-03-25 03:00:00 (UTC+8).
- Number of distinct (cmdb_id, kpi_name) pairs observed in that window: 1,764.
- Number of pairs that had at least one point >= their global P95: 1,764 (i.e., every pair in the window).
- Top 20 pairs (by anomaly_count then points_in_window) show many pairs where every point in the window is >= global P95 (points_in_window = 31 and anomaly_count = 31). Examples from the top results:
  - IG01 / JVM-Memory_7778_JVM_Memory_HeapMemoryMax — global_p95 = 2,040,135,680; 31 points in window; anomaly_count = 31; earliest anomaly = 2021-03-25 02:30:00; max = 2,040,135,680.
  - IG01 / JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed — global_p95 = 58,461,192; 31 points; anomaly_count = 31.
  - IG02 / JVM-Memory_7778_JVM_Memory_HeapMemoryMax — same heap max value; 31/31 anomalies.
  - MG01 / JVM-Memory_7779_JVM_Memory_HeapMemoryMax — same pattern.
  - Several MG01/MG02 OSLinux process/network KPIs with global_p95 = 0 or small numbers and anomaly_count = 31 (earliest anomaly 02:30:00).
  - Tomcat01 request KPIs such as ErrorCountRequestInfo (global_p95 = 1051) and MaxTimeRequestInfo (global_p95 = 397,808) also show 31/31 anomalies.

- Interpretation / caveat: The threshold used is the global P95 computed over the entire available series for each pair, and anomalies were counted as value >= global_p95. Because the check is inclusive (>=) and some global P95 values equal regularly-observed values (including zeros), many series naturally meet or exceed that threshold for every point in the incident window — resulting in all 1,764 pairs being flagged. Earliest anomalies for top pairs occur at the start of the window (02:30:00).

If you want, I can:
- Recompute anomalies using a strict > global P95 rule to reduce ties,
- Exclude KPIs whose global P95 equals a typical constant (e.g., 0) before flagging,
- Or return the full top-20 table again with brief explanations per KPI.

The original code execution output of IPython Kernel is also provided below for reference:

{'window_start': '2021-03-25 02:30:00 CST', 'window_end': '2021-03-25 03:00:00 CST', 'total_pairs_in_window': 1764, 'total_pairs_with_anomalies': 1764, 'top_20_pairs': [{'cmdb_id': 'IG01', 'kpi_name': 'JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'global_p95': 2040135680.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 2040135680.0}, {'cmdb_id': 'IG01', 'kpi_name': 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'global_p95': 58461192.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 58461192.0}, {'cmdb_id': 'IG02', 'kpi_name': 'JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'global_p95': 2040135680.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 2040135680.0}, {'cmdb_id': 'IG02', 'kpi_name': 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'global_p95': 58605336.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 58605336.0}, {'cmdb_id': 'MG01', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'global_p95': 2040135680.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 2040135680.0}, {'cmdb_id': 'MG01', 'kpi_name': 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'global_p95': 0.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 0.0}, {'cmdb_id': 'MG01', 'kpi_name': 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'global_p95': 0.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 0.0}, {'cmdb_id': 'MG01', 'kpi_name': 'OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies', 'global_p95': 2.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 2.0}, {'cmdb_id': 'MG01', 'kpi_name': 'OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc', 'global_p95': 0.3, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 0.3}, {'cmdb_id': 'MG01', 'kpi_name': 'OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount', 'global_p95': 3.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 3.0}, {'cmdb_id': 'MG02', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'global_p95': 2040135680.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 2040135680.0}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'global_p95': 0.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 0.0}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'global_p95': 0.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 0.0}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies', 'global_p95': 1.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 1.0}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc', 'global_p95': 0.3, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 0.3}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount', 'global_p95': 3.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 3.0}, {'cmdb_id': 'Tomcat01', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_ErrorCountRequestInfo', 'global_p95': 1051.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 1051.0}, {'cmdb_id': 'Tomcat01', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo', 'global_p95': 397808.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 397808.0}, {'cmdb_id': 'Tomcat01', 'kpi_name': 'Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter', 'global_p95': 0.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 0.0}, {'cmdb_id': 'Tomcat01', 'kpi_name': 'Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCounter', 'global_p95': 0.0, 'points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp': '2021-03-25 02:30:00 CST', 'max_value_in_window': 0.0}]}```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

sh = pytz.timezone('Asia/Shanghai')

# Prepare data
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_metric['ts_sh'] = df_metric['ts_utc'].dt.tz_convert(sh)

# 1) Global P95 per (cmdb_id, kpi_name) using full series
global_p95 = (
    df_metric
    .dropna(subset=['value'])
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter to incident window (inclusive) in UTC+8
window_start_sh = pd.Timestamp('2021-03-25 02:30:00').tz_localize(sh)
window_end_sh = pd.Timestamp('2021-03-25 03:00:00').tz_localize(sh)
df_window = df_metric[(df_metric['ts_sh'] >= window_start_sh) & (df_metric['ts_sh'] <= window_end_sh)].copy()

# 3) Merge global_p95 and mark strict anomalies (value > global_p95)
df_win = df_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
df_win['is_anom_strict'] = df_win['value'] > df_win['global_p95']

# Helper to compute longest consecutive run of strict anomalies for a group
def compute_runs(g):
    # g sorted by timestamp
    g = g.sort_values('timestamp').reset_index(drop=True)
    pts = int(g.shape[0])
    anomaly_count = int(g['is_anom_strict'].sum())
    max_value = g['value'].max() if pts > 0 else None

    if anomaly_count == 0:
        return pd.Series({
            'points_in_window': pts,
            'anomaly_count_strict': anomaly_count,
            'earliest_anomaly_timestamp_strict': pd.NaT,
            'max_value_in_window': max_value,
            'longest_consecutive_anomalous_run': 0,
            'start_timestamp_of_longest_run': pd.NaT
        })

    # identify consecutive minutes: difference of timestamp == 60 seconds
    g['prev_ts'] = g['timestamp'].shift(1)
    g['ts_gap'] = (g['timestamp'] - g['prev_ts']).fillna(9999).astype(int)
    # we'll iterate rows to find consecutive runs where is_anom_strict == True and consecutive by 60s
    longest_run = 0
    longest_start_ts = None
    current_run = 0
    current_start_ts = None

    for idx, row in g.iterrows():
        if row['is_anom_strict']:
            if current_run == 0:
                # start new run
                current_run = 1
                current_start_ts = row['ts_sh']
            else:
                # check if contiguous minute from previous row (previous timestamp +60)
                prev_row = g.iloc[idx-1]
                if (row['timestamp'] - prev_row['timestamp']) == 60 and prev_row['is_anom_strict']:
                    current_run += 1
                else:
                    # gap or previous not anomaly: restart
                    current_run = 1
                    current_start_ts = row['ts_sh']
            # update longest
            if current_run > longest_run:
                longest_run = current_run
                longest_start_ts = current_start_ts
        else:
            current_run = 0
            current_start_ts = None

    # earliest anomaly ts (first occurrence)
    earliest_idx = g[g['is_anom_strict']].index[0]
    earliest_ts = g.loc[earliest_idx, 'ts_sh']

    return pd.Series({
        'points_in_window': pts,
        'anomaly_count_strict': anomaly_count,
        'earliest_anomaly_timestamp_strict': earliest_ts,
        'max_value_in_window': max_value,
        'longest_consecutive_anomalous_run': longest_run,
        'start_timestamp_of_longest_run': longest_start_ts
    })

# 4) Group and compute runs
runs_df = df_win.groupby(['cmdb_id', 'kpi_name']).apply(compute_runs).reset_index()

# Merge global_p95 to runs_df
summary_df = runs_df.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Format timestamps to strings (UTC+8) or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(sh).strftime('%Y-%m-%d %H:%M:%S %Z')

summary_df['earliest_anomaly_timestamp_strict'] = summary_df['earliest_anomaly_timestamp_strict'].apply(fmt_ts)
summary_df['start_timestamp_of_longest_run'] = summary_df['start_timestamp_of_longest_run'].apply(fmt_ts)

# Ensure integer types
summary_df['points_in_window'] = summary_df['points_in_window'].astype(int)
summary_df['anomaly_count_strict'] = summary_df['anomaly_count_strict'].astype(int)
summary_df['longest_consecutive_anomalous_run'] = summary_df['longest_consecutive_anomalous_run'].astype(int)

# 5) Sort by longest_consecutive_anomalous_run desc, then anomaly_count_strict desc. Take top 20.
summary_sorted = summary_df.sort_values(['longest_consecutive_anomalous_run', 'anomaly_count_strict'], ascending=[False, False]).reset_index(drop=True)
top20 = summary_sorted.head(20).copy()

# Select and order required columns
top20 = top20[[
    'cmdb_id',
    'kpi_name',
    'global_p95',
    'points_in_window',
    'anomaly_count_strict',
    'earliest_anomaly_timestamp_strict',
    'max_value_in_window',
    'longest_consecutive_anomalous_run',
    'start_timestamp_of_longest_run'
]]

# 6) Count pairs with longest_consecutive_anomalous_run >= 3 (candidate faults)
candidate_faults_count = int((summary_df['longest_consecutive_anomalous_run'] >= 3).sum())

# Prepare JSON-friendly result
result = {
    "window_start": window_start_sh.strftime('%Y-%m-%d %H:%M:%S %Z'),
    "window_end": window_end_sh.strftime('%Y-%m-%d %H:%M:%S %Z'),
    "total_pairs_in_window": int(summary_df.shape[0]),
    "candidate_faults_count_longest_run_ge_3": candidate_faults_count,
    "top_20_pairs_by_longest_run": top20.to_dict(orient='records')
}

result

```
Out[3]:
```
```python
# Plain-English summary of the strict (> P95) anomaly detection results
summary = (
    "Summary of strict (> global P95) metric anomaly detection (window 2021-03-25 02:30:00 to 03:00:00 UTC+8):\n\n"
    "- Total distinct (cmdb_id, kpi_name) pairs in the window: 1,764.\n"
    "- Candidate faults (longest consecutive run >= 3 minutes): 0 (none).\n\n"
    "- With the stricter rule (value > global P95) anomalies are much rarer. "
    "Top pairs have anomaly_count_strict = 2 and longest_consecutive_anomalous_run = 2 (i.e., two consecutive minutes) rather than long sustained runs.\n\n"
    "- Representative top pairs (each shows 2 strict anomalies and longest run = 2):\n"
    "  * IG01 / JVM-Memory_7778_JVM_Memory_HeapMemoryUsed — global_p95 ~ 456,229,532; 31 points; 2 anomalies; earliest strict anomaly 2021-03-25 02:59:00; max ~ 461,224,760.\n"
    "  * IG02 / JVM-Memory_7778_JVM_Memory_HeapMemoryUsed — similar pattern; earliest strict anomaly 02:59:00.\n"
    "  * MG01 / JVM-Memory_7779_JVM_Memory_HeapMemoryUsed — 2 strict anomalies; longest run 2 starting 02:50:00.\n"
    "  * Several Mysql02 KPIs (Innodb fsyncs, ThreadsRunning, etc.) show 2 strict anomalies around 02:44:00 with max values exceeding P95.\n"
    "  * Various OSLinux CPU/uptime KPIs on MG02, Redis02, Tomcat01 also show brief 2-minute strict anomalies at different timestamps.\n\n"
    "- Interpretation: Using a strict '>' rule removed the many ties from the inclusive check; remaining anomalies are short (at most 2 consecutive minutes) and isolated across multiple components. "
    "No component-KPI exhibits a sustained (>=3 minute) strict anomaly run during this incident window.\n\n"
    "If you want next steps, I can: re-evaluate using different percentile thresholds (e.g., P90), require longer minimum run length for faults, or produce a CSV/JSON of all pairs with their runs for deeper inspection."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'window_start': '2021-03-25 02:30:00 CST', 'window_end': '2021-03-25 03:00:00 CST', 'total_pairs_in_window': 1764, 'candidate_faults_count_longest_run_ge_3': 0, 'top_20_pairs_by_longest_run': [{'cmdb_id': 'IG01', 'kpi_name': 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'global_p95': 456229532.0, 'points_in_window': 31, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:59:00 CST', 'max_value_in_window': 461224760.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:59:00 CST'}, {'cmdb_id': 'IG02', 'kpi_name': 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'global_p95': 355451100.0, 'points_in_window': 31, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:59:00 CST', 'max_value_in_window': 359770016.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:59:00 CST'}, {'cmdb_id': 'MG01', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'global_p95': 1004222544.0, 'points_in_window': 31, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:50:00 CST', 'max_value_in_window': 1023528400.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:50:00 CST'}, {'cmdb_id': 'MG01', 'kpi_name': 'OSLinux-OSLinux_ZABBIX_Host_Uptime', 'global_p95': 7812012.5, 'points_in_window': 31, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:59:00 CST', 'max_value_in_window': 7812103.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:59:00 CST'}, {'cmdb_id': 'MG02', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'global_p95': 940006020.0, 'points_in_window': 31, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:33:00 CST', 'max_value_in_window': 980341688.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:33:00 CST'}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-CPU_CPU_CPULoad', 'global_p95': 2.01, 'points_in_window': 31, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:46:00 CST', 'max_value_in_window': 2.16, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:46:00 CST'}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-CPU_CPU_CPUWio', 'global_p95': 23.1854, 'points_in_window': 31, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:44:00 CST', 'max_value_in_window': 23.4746, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:44:00 CST'}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-OSLinux_ZABBIX_Host_Uptime', 'global_p95': 7811421.5, 'points_in_window': 31, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:59:00 CST', 'max_value_in_window': 7811511.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:59:00 CST'}, {'cmdb_id': 'Mysql02', 'kpi_name': 'Mysql-MySQL_3306_Innodb data pending fsyncs', 'global_p95': 0.5999999999999979, 'points_in_window': 29, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:44:00 CST', 'max_value_in_window': 1.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:44:00 CST'}, {'cmdb_id': 'Mysql02', 'kpi_name': 'Mysql-MySQL_3306_Innodb os log pending fsyncs', 'global_p95': 0.5999999999999979, 'points_in_window': 29, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:44:00 CST', 'max_value_in_window': 1.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:44:00 CST'}, {'cmdb_id': 'Mysql02', 'kpi_name': 'Mysql-MySQL_3306_MaxTrxRowsModified', 'global_p95': 0.5999999999999979, 'points_in_window': 29, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:44:00 CST', 'max_value_in_window': 1.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:44:00 CST'}, {'cmdb_id': 'Mysql02', 'kpi_name': 'Mysql-MySQL_3306_ThreadsRunning', 'global_p95': 1.5999999999999979, 'points_in_window': 29, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:44:00 CST', 'max_value_in_window': 4.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:44:00 CST'}, {'cmdb_id': 'Mysql02', 'kpi_name': 'Mysql-MySQL_3306_max trx lock memory bytes', 'global_p95': 681.5999999999976, 'points_in_window': 29, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:44:00 CST', 'max_value_in_window': 1136.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:44:00 CST'}, {'cmdb_id': 'Redis02', 'kpi_name': 'OSLinux-CPU_CPU_CPULoad', 'global_p95': 0.10399999999999979, 'points_in_window': 30, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:34:00 CST', 'max_value_in_window': 0.4, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:34:00 CST'}, {'cmdb_id': 'Redis02', 'kpi_name': 'OSLinux-CPU_CPU_CPUWio', 'global_p95': 0.08624989999999993, 'points_in_window': 30, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:46:00 CST', 'max_value_in_window': 0.108036, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:46:00 CST'}, {'cmdb_id': 'Tomcat01', 'kpi_name': 'OSLinux-CPU_CPU_CPULoad', 'global_p95': 1.1084999999999998, 'points_in_window': 30, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:56:00 CST', 'max_value_in_window': 1.16, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:56:00 CST'}, {'cmdb_id': 'Tomcat01', 'kpi_name': 'OSLinux-OSLinux_ZABBIX_Host_Uptime', 'global_p95': 7811309.55, 'points_in_window': 30, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:59:00 CST', 'max_value_in_window': 7811396.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:59:00 CST'}, {'cmdb_id': 'Tomcat01', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo', 'global_p95': 2123821088.5, 'points_in_window': 31, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:59:00 CST', 'max_value_in_window': 2123821228.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:59:00 CST'}, {'cmdb_id': 'Tomcat01', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_RequestCountRequestInfo', 'global_p95': 3208039.5, 'points_in_window': 31, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:59:00 CST', 'max_value_in_window': 3208041.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:59:00 CST'}, {'cmdb_id': 'Tomcat02', 'kpi_name': 'OSLinux-OSLinux_ZABBIX_Host_Uptime', 'global_p95': 7811244.0, 'points_in_window': 31, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:59:00 CST', 'max_value_in_window': 7811334.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:59:00 CST'}]}```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse df_trace if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

sh = pytz.timezone('Asia/Shanghai')

# Ensure numeric
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Timestamp conversions (UTC then to Asia/Shanghai)
df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_trace['ts_sh'] = df_trace['ts_utc'].dt.tz_convert(sh)

# 1) Compute global P95 per (cmdb_id, trace_name) using full series (drop NaNs)
trace_global_p95 = (
    df_trace
    .dropna(subset=['value'])
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter to incident window (inclusive) in UTC+8
window_start_sh = pd.Timestamp('2021-03-25 02:30:00').tz_localize(sh)
window_end_sh = pd.Timestamp('2021-03-25 03:00:00').tz_localize(sh)
df_window = df_trace[(df_trace['ts_sh'] >= window_start_sh) & (df_trace['ts_sh'] <= window_end_sh)].copy()

# 3) Merge global_p95 and mark strict anomalies (value > global_p95)
df_win = df_window.merge(trace_global_p95, on=['cmdb_id', 'trace_name'], how='left')
df_win['is_anom_strict'] = df_win['value'] > df_win['global_p95']

# Helper to compute longest consecutive run of strict anomalies for a group
def compute_runs_trace(g):
    g = g.sort_values('timestamp').reset_index(drop=True)
    pts = int(g.shape[0])
    anomaly_count = int(g['is_anom_strict'].sum())
    max_value = g['value'].max() if pts > 0 else None

    if anomaly_count == 0:
        return pd.Series({
            'points_in_window': pts,
            'anomaly_count_strict': anomaly_count,
            'earliest_anomaly_timestamp_strict': pd.NaT,
            'max_value_in_window': max_value,
            'longest_consecutive_anomalous_run': 0,
            'start_timestamp_of_longest_run': pd.NaT
        })

    longest_run = 0
    longest_start_ts = None
    current_run = 0
    current_start_ts = None

    for idx, row in g.iterrows():
        if row['is_anom_strict']:
            if current_run == 0:
                current_run = 1
                current_start_ts = row['ts_sh']
            else:
                prev_row = g.iloc[idx-1]
                # require exact 60-second adjacency for consecutive minutes
                if (row['timestamp'] - prev_row['timestamp']) == 60 and prev_row['is_anom_strict']:
                    current_run += 1
                else:
                    current_run = 1
                    current_start_ts = row['ts_sh']
            if current_run > longest_run:
                longest_run = current_run
                longest_start_ts = current_start_ts
        else:
            current_run = 0
            current_start_ts = None

    earliest_idx = g[g['is_anom_strict']].index[0]
    earliest_ts = g.loc[earliest_idx, 'ts_sh']

    return pd.Series({
        'points_in_window': pts,
        'anomaly_count_strict': anomaly_count,
        'earliest_anomaly_timestamp_strict': earliest_ts,
        'max_value_in_window': max_value,
        'longest_consecutive_anomalous_run': longest_run,
        'start_timestamp_of_longest_run': longest_start_ts
    })

# 4) Group by pair and compute runs
runs_trace_df = df_win.groupby(['cmdb_id', 'trace_name']).apply(compute_runs_trace).reset_index()

# Merge global_p95
summary_trace_df = runs_trace_df.merge(trace_global_p95, on=['cmdb_id', 'trace_name'], how='left')

# Format timestamps to strings (UTC+8) or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(sh).strftime('%Y-%m-%d %H:%M:%S %Z')

summary_trace_df['earliest_anomaly_timestamp_strict'] = summary_trace_df['earliest_anomaly_timestamp_strict'].apply(fmt_ts)
summary_trace_df['start_timestamp_of_longest_run'] = summary_trace_df['start_timestamp_of_longest_run'].apply(fmt_ts)

# Ensure integer types
summary_trace_df['points_in_window'] = summary_trace_df['points_in_window'].astype(int)
summary_trace_df['anomaly_count_strict'] = summary_trace_df['anomaly_count_strict'].astype(int)
summary_trace_df['longest_consecutive_anomalous_run'] = summary_trace_df['longest_consecutive_anomalous_run'].astype(int)

# 5) Sort and select top 20
summary_sorted = summary_trace_df.sort_values(['longest_consecutive_anomalous_run', 'anomaly_count_strict'], ascending=[False, False]).reset_index(drop=True)
top20_trace = summary_sorted.head(20)[[
    'cmdb_id',
    'trace_name',
    'global_p95',
    'points_in_window',
    'anomaly_count_strict',
    'earliest_anomaly_timestamp_strict',
    'max_value_in_window',
    'longest_consecutive_anomalous_run',
    'start_timestamp_of_longest_run'
]].copy()

# 6) Count candidate faulty traces with longest run >= 3
candidate_faults_count_trace = int((summary_trace_df['longest_consecutive_anomalous_run'] >= 3).sum())

# Display compact outputs (DataFrame and count)
top20_trace, candidate_faults_count_trace

```
Out[4]:
```
Summary (plain English):

- Analysis window: 2021-03-25 02:30:00 to 03:00:00 (UTC+8). Global P95 thresholds were computed per (cmdb_id, trace_name) using the full available trace series, and points were flagged only when value > global P95 (strict).

- Key outcome: No trace pair showed a sustained strict anomaly of 3 minutes or more. Candidate faulty traces with longest consecutive anomalous run >= 3: 0.

- Top findings (brief):
  - IG01 / trace.self.row_count — global P95 = 62.2; 27 points in window; 2 strict anomalies; earliest strict anomaly 02:42; longest consecutive run = 2 minutes; max = 74.
  - IG02 / trace.self.row_count — global P95 = 61.6; 27 points; 2 strict anomalies at/around 02:42; longest run = 2 minutes.
  - Other top short runs (longest run = 2 minutes; anomaly_count = 2) occur for traces between IG/MG/Tomcat/docker components, e.g. trace.to_Tomcat01.duration_mean, trace.to_dockerB1.row_count, trace.from_Tomcat01.duration_mean/row_count, and several Tomcat trace.self.* and trace.to_MG02 metrics. Many earliest strict anomalies cluster around ~02:42 and ~02:57–02:58.

- Interpretation: Trace-level anomalies are short (mostly 1–2 consecutive minutes) and scattered across IG/MG/Tomcat/docker traces. There is no sustained (> = 3 min) trace anomaly during this incident window.

- Suggested next steps: inspect traces around the highlighted timestamps (notably ~02:42 and ~02:57–02:58) for the top components (IG01/IG02, MG01/MG02, Tomcat01) to identify correlated request paths or spikes in latency/row_count that could explain short-lived degradations.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name  global_p95  points_in_window  anomaly_count_strict earliest_anomaly_timestamp_strict  max_value_in_window  longest_consecutive_anomalous_run start_timestamp_of_longest_run
0       IG01               trace.self.row_count   62.200000                27                     2           2021-03-25 02:42:00 CST            74.000000                                  2        2021-03-25 02:42:00 CST
1       IG02               trace.self.row_count   61.600000                27                     2           2021-03-25 02:42:00 CST            74.000000                                  2        2021-03-25 02:42:00 CST
2       IG02    trace.to_Tomcat01.duration_mean    0.086150                24                     2           2021-03-25 02:58:00 CST             0.093500                                  2        2021-03-25 02:58:00 CST
3       MG01        trace.to_dockerB1.row_count   10.950000                22                     2           2021-03-25 02:42:00 CST            12.000000                                  2        2021-03-25 02:42:00 CST
4       MG02  trace.from_Tomcat01.duration_mean    0.068200                24                     2           2021-03-25 02:57:00 CST             0.087500                                  2        2021-03-25 02:57:00 CST
5       MG02      trace.from_Tomcat01.row_count   11.400000                24                     2           2021-03-25 02:42:00 CST            15.000000                                  2        2021-03-25 02:42:00 CST
6   Tomcat01      trace.from_IG02.duration_mean    0.086150                24                     2           2021-03-25 02:58:00 CST             0.093500                                  2        2021-03-25 02:58:00 CST
7   Tomcat01           trace.self.duration_mean    0.013416                26                     2           2021-03-25 02:58:00 CST             0.014917                                  2        2021-03-25 02:58:00 CST
8   Tomcat01        trace.to_MG02.duration_mean    0.068200                24                     2           2021-03-25 02:57:00 CST             0.087500                                  2        2021-03-25 02:57:00 CST
9   Tomcat01            trace.to_MG02.row_count   11.400000                24                     2           2021-03-25 02:42:00 CST            15.000000                                  2        2021-03-25 02:42:00 CST
10  Tomcat02            trace.self.duration_p95    0.121207                24                     2           2021-03-25 02:57:00 CST             0.209250                                  2        2021-03-25 02:57:00 CST
11  Tomcat03               trace.self.row_count  127.200000                23                     2           2021-03-25 02:42:00 CST           132.000000                                  2        2021-03-25 02:42:00 CST
12  dockerB1          trace.from_MG01.row_count   10.950000                22                     2           2021-03-25 02:42:00 CST            12.000000                                  2        2021-03-25 02:42:00 CST
13  dockerB1            trace.self.duration_p95    0.010163                26                     2           2021-03-25 02:40:00 CST             0.010450                                  2        2021-03-25 02:40:00 CST
14      IG01           trace.self.duration_mean    0.173922                27                     2           2021-03-25 02:34:00 CST             0.285000                                  1        2021-03-25 02:34:00 CST
15      IG01            trace.self.duration_p95    0.539070                27                     2           2021-03-25 02:34:00 CST             0.896000                                  1        2021-03-25 02:34:00 CST
16      IG01    trace.to_Tomcat02.duration_mean    0.229250                23                     2           2021-03-25 02:41:00 CST             0.283000                                  1        2021-03-25 02:41:00 CST
17      IG01     trace.to_Tomcat02.duration_p95    0.282620                23                     2           2021-03-25 02:41:00 CST             0.376250                                  1        2021-03-25 02:41:00 CST
18      IG01        trace.to_Tomcat02.row_count    8.800000                23                     2           2021-03-25 02:42:00 CST            11.000000                                  1        2021-03-25 02:42:00 CST
19      IG02           trace.self.duration_mean    0.101377                27                     2           2021-03-25 02:34:00 CST             0.111000                                  1        2021-03-25 02:34:00 CST, 0)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# reuse df_log if available; otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

sh = pytz.timezone('Asia/Shanghai')

# Ensure numeric values and timestamps
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
df_log['ts_utc'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_log['ts_sh'] = df_log['ts_utc'].dt.tz_convert(sh)

# 1) Global P95 per (cmdb_id, log_name) using full series (do NOT filter first)
global_p95_log = (
    df_log
    .dropna(subset=['value'])
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter to incident window (inclusive) in UTC+8
# reuse window_start_sh/window_end_sh if present; otherwise define
try:
    window_start_sh, window_end_sh
except NameError:
    window_start_sh = pd.Timestamp('2021-03-25 02:30:00').tz_localize(sh)
    window_end_sh = pd.Timestamp('2021-03-25 03:00:00').tz_localize(sh)

df_window_log = df_log[(df_log['ts_sh'] >= window_start_sh) & (df_log['ts_sh'] <= window_end_sh)].copy()

# 3) Merge global_p95 and mark strict anomalies (value > global_p95)
df_win_log = df_window_log.merge(global_p95_log, on=['cmdb_id', 'log_name'], how='left')
df_win_log['is_anom_strict'] = df_win_log['value'] > df_win_log['global_p95']

# 4) For each pair compute summary including longest consecutive anomalous run
def compute_runs_log(g):
    g = g.sort_values('timestamp').reset_index(drop=True)
    pts = int(g.shape[0])
    anomaly_count = int(g['is_anom_strict'].sum())
    max_value = g['value'].max() if pts > 0 else None

    if anomaly_count == 0:
        return pd.Series({
            'points_in_window': pts,
            'anomaly_count_strict': anomaly_count,
            'earliest_anomaly_timestamp_strict': pd.NaT,
            'max_value_in_window': max_value,
            'longest_consecutive_anomalous_run': 0,
            'start_timestamp_of_longest_run': pd.NaT
        })

    longest_run = 0
    longest_start_ts = None
    current_run = 0
    current_start_ts = None

    for idx, row in g.iterrows():
        if row['is_anom_strict']:
            if current_run == 0:
                current_run = 1
                current_start_ts = row['ts_sh']
            else:
                prev_row = g.iloc[idx-1]
                # require exact 60-second adjacency for consecutive minutes
                if (row['timestamp'] - prev_row['timestamp']) == 60 and prev_row['is_anom_strict']:
                    current_run += 1
                else:
                    current_run = 1
                    current_start_ts = row['ts_sh']
            if current_run > longest_run:
                longest_run = current_run
                longest_start_ts = current_start_ts
        else:
            current_run = 0
            current_start_ts = None

    earliest_idx = g[g['is_anom_strict']].index[0]
    earliest_ts = g.loc[earliest_idx, 'ts_sh']

    return pd.Series({
        'points_in_window': pts,
        'anomaly_count_strict': anomaly_count,
        'earliest_anomaly_timestamp_strict': earliest_ts,
        'max_value_in_window': max_value,
        'longest_consecutive_anomalous_run': longest_run,
        'start_timestamp_of_longest_run': longest_start_ts
    })

runs_log_df = df_win_log.groupby(['cmdb_id', 'log_name']).apply(compute_runs_log).reset_index()

# Merge global_p95 into runs
summary_log_df = runs_log_df.merge(global_p95_log, on=['cmdb_id', 'log_name'], how='left')

# Format timestamps to strings (UTC+8) or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(sh).strftime('%Y-%m-%d %H:%M:%S %Z')

summary_log_df['earliest_anomaly_timestamp_strict'] = summary_log_df['earliest_anomaly_timestamp_strict'].apply(fmt_ts)
summary_log_df['start_timestamp_of_longest_run'] = summary_log_df['start_timestamp_of_longest_run'].apply(fmt_ts)

# Ensure integer types for counts
summary_log_df['points_in_window'] = summary_log_df['points_in_window'].astype(int)
summary_log_df['anomaly_count_strict'] = summary_log_df['anomaly_count_strict'].astype(int)
summary_log_df['longest_consecutive_anomalous_run'] = summary_log_df['longest_consecutive_anomalous_run'].astype(int)

# 5) Sort and select top 20 pairs
summary_sorted_log = summary_log_df.sort_values(['longest_consecutive_anomalous_run', 'anomaly_count_strict'], ascending=[False, False]).reset_index(drop=True)
top20_log = summary_sorted_log.head(20)[[
    'cmdb_id',
    'log_name',
    'global_p95',
    'points_in_window',
    'anomaly_count_strict',
    'earliest_anomaly_timestamp_strict',
    'max_value_in_window',
    'longest_consecutive_anomalous_run',
    'start_timestamp_of_longest_run'
]].copy()

# Count candidate log faults (longest run >= 3)
candidate_faults_count_log = int((summary_log_df['longest_consecutive_anomalous_run'] >= 3).sum())

# 6) If candidate faults exist, collect up to 10 raw rows per candidate within window (timestamp and value)
candidate_samples = {}
if candidate_faults_count_log > 0:
    candidates = summary_log_df[summary_log_df['longest_consecutive_anomalous_run'] >= 3][['cmdb_id', 'log_name']]
    for _, row in candidates.iterrows():
        cm = row['cmdb_id']; ln = row['log_name']
        subset = df_win_log[(df_win_log['cmdb_id']==cm) & (df_win_log['log_name']==ln)].sort_values('timestamp')[['timestamp','ts_sh','value']].head(10)
        # format timestamp string in UTC+8
        subset_display = subset.copy()
        subset_display['timestamp_sh'] = subset_display['ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        candidate_samples[(cm, ln)] = subset_display[['timestamp_sh','value']].to_dict(orient='records')

# Prepare compact JSON-friendly result
result = {
    "window_start": window_start_sh.strftime('%Y-%m-%d %H:%M:%S %Z'),
    "window_end": window_end_sh.strftime('%Y-%m-%d %H:%M:%S %Z'),
    "total_pairs_in_window": int(summary_log_df.shape[0]),
    "candidate_log_faults_count_longest_run_ge_3": candidate_faults_count_log,
    "top_20_pairs_by_longest_run": top20_log.to_dict(orient='records'),
    "candidate_samples_up_to_10_rows_per_candidate": candidate_samples  # will be empty if no candidates
}

result

```
Out[5]:
```
```python
# Plain-English summary of log-level anomaly scan (window 2021-03-25 02:30:00 to 03:00:00 UTC+8)
summary = (
    "Log-level anomaly scan (strict > global P95) results:\n\n"
    "- Window: 2021-03-25 02:30:00 to 2021-03-25 03:00:00 (UTC+8).\n"
    "- Total distinct (cmdb_id, log_name) pairs in window: 15.\n"
    "- Candidate log faults (longest consecutive anomalous run >= 3 minutes): 0 (none).\n\n"
    "- Top short anomalies (longest run = 2 minutes):\n"
    "  * Tomcat03 / log.row_count — global_p95 = 18.8; 23 points; 2 strict anomalies; earliest 2021-03-25 02:42; max = 19.\n"
    "  * apache02 / log.row_count — global_p95 = 37.1; 23 points; 2 strict anomalies; earliest 2021-03-25 02:42; max = 41.\n\n"
    "- Many other log pairs (Tomcat and apache instances) show at most 1–2 isolated strict anomalies or none (e.g., log.error_count is 0 across many pairs).\n"
    "- error_logs.csv is empty, so there are no raw error log messages to inspect.\n\n"
    "Conclusion: No sustained log-level faults (>=3 consecutive minutes) were detected in the incident window. The notable findings are brief 1–2 minute spikes in log row_count for Tomcat03 and apache02 around ~02:42. "
    "If you want, I can return the raw log rows for those top pairs or run a less strict threshold (e.g., P90) to surface more signals."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'window_start': '2021-03-25 02:30:00 CST', 'window_end': '2021-03-25 03:00:00 CST', 'total_pairs_in_window': 15, 'candidate_log_faults_count_longest_run_ge_3': 0, 'top_20_pairs_by_longest_run': [{'cmdb_id': 'Tomcat03', 'log_name': 'log.row_count', 'global_p95': 18.799999999999997, 'points_in_window': 23, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:42:00 CST', 'max_value_in_window': 19.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:42:00 CST'}, {'cmdb_id': 'apache02', 'log_name': 'log.row_count', 'global_p95': 37.09999999999999, 'points_in_window': 23, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:42:00 CST', 'max_value_in_window': 41.0, 'longest_consecutive_anomalous_run': 2, 'start_timestamp_of_longest_run': '2021-03-25 02:42:00 CST'}, {'cmdb_id': 'Tomcat01', 'log_name': 'log.row_count', 'global_p95': 19.75, 'points_in_window': 26, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:37:00 CST', 'max_value_in_window': 22.0, 'longest_consecutive_anomalous_run': 1, 'start_timestamp_of_longest_run': '2021-03-25 02:37:00 CST'}, {'cmdb_id': 'Tomcat02', 'log_name': 'log.row_count', 'global_p95': 15.699999999999996, 'points_in_window': 24, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:42:00 CST', 'max_value_in_window': 18.0, 'longest_consecutive_anomalous_run': 1, 'start_timestamp_of_longest_run': '2021-03-25 02:42:00 CST'}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.row_count', 'global_p95': 16.749999999999996, 'points_in_window': 22, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:35:00 CST', 'max_value_in_window': 18.0, 'longest_consecutive_anomalous_run': 1, 'start_timestamp_of_longest_run': '2021-03-25 02:35:00 CST'}, {'cmdb_id': 'apache01', 'log_name': 'log.row_count', 'global_p95': 29.799999999999997, 'points_in_window': 27, 'anomaly_count_strict': 2, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:42:00 CST', 'max_value_in_window': 33.0, 'longest_consecutive_anomalous_run': 1, 'start_timestamp_of_longest_run': '2021-03-25 02:42:00 CST'}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.gc_pause_max', 'global_p95': 0.089, 'points_in_window': 2, 'anomaly_count_strict': 1, 'earliest_anomaly_timestamp_strict': '2021-03-25 02:48:00 CST', 'max_value_in_window': 0.09, 'longest_consecutive_anomalous_run': 1, 'start_timestamp_of_longest_run': '2021-03-25 02:48:00 CST'}, {'cmdb_id': 'Tomcat01', 'log_name': 'log.error_count', 'global_p95': 0.0, 'points_in_window': 26, 'anomaly_count_strict': 0, 'earliest_anomaly_timestamp_strict': None, 'max_value_in_window': 0.0, 'longest_consecutive_anomalous_run': 0, 'start_timestamp_of_longest_run': None}, {'cmdb_id': 'Tomcat01', 'log_name': 'log.gc_pause_max', 'global_p95': 0.07, 'points_in_window': 1, 'anomaly_count_strict': 0, 'earliest_anomaly_timestamp_strict': None, 'max_value_in_window': 0.07, 'longest_consecutive_anomalous_run': 0, 'start_timestamp_of_longest_run': None}, {'cmdb_id': 'Tomcat02', 'log_name': 'log.error_count', 'global_p95': 0.0, 'points_in_window': 24, 'anomaly_count_strict': 0, 'earliest_anomaly_timestamp_strict': None, 'max_value_in_window': 0.0, 'longest_consecutive_anomalous_run': 0, 'start_timestamp_of_longest_run': None}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.error_count', 'global_p95': 0.0, 'points_in_window': 23, 'anomaly_count_strict': 0, 'earliest_anomaly_timestamp_strict': None, 'max_value_in_window': 0.0, 'longest_consecutive_anomalous_run': 0, 'start_timestamp_of_longest_run': None}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.gc_pause_max', 'global_p95': 0.08, 'points_in_window': 1, 'anomaly_count_strict': 0, 'earliest_anomaly_timestamp_strict': None, 'max_value_in_window': 0.08, 'longest_consecutive_anomalous_run': 0, 'start_timestamp_of_longest_run': None}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.error_count', 'global_p95': 0.0, 'points_in_window': 22, 'anomaly_count_strict': 0, 'earliest_anomaly_timestamp_strict': None, 'max_value_in_window': 0.0, 'longest_consecutive_anomalous_run': 0, 'start_timestamp_of_longest_run': None}, {'cmdb_id': 'apache01', 'log_name': 'log.error_count', 'global_p95': 0.0, 'points_in_window': 27, 'anomaly_count_strict': 0, 'earliest_anomaly_timestamp_strict': None, 'max_value_in_window': 0.0, 'longest_consecutive_anomalous_run': 0, 'start_timestamp_of_longest_run': None}, {'cmdb_id': 'apache02', 'log_name': 'log.error_count', 'global_p95': 0.0, 'points_in_window': 23, 'anomaly_count_strict': 0, 'earliest_anomaly_timestamp_strict': None, 'max_value_in_window': 0.0, 'longest_consecutive_anomalous_run': 0, 'start_timestamp_of_longest_run': None}], 'candidate_samples_up_to_10_rows_per_candidate': {}}```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse existing df_metric and timezone variables if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

sh = pytz.timezone('Asia/Shanghai')

# Ensure numeric and timestamp columns exist (reuse if already computed)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'ts_utc' not in df_metric.columns:
    df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
if 'ts_sh' not in df_metric.columns:
    df_metric['ts_sh'] = df_metric['ts_utc'].dt.tz_convert(sh)

# 1) Compute global P90 per (cmdb_id, kpi_name) on full series (do NOT filter first)
global_p90 = (
    df_metric
    .dropna(subset=['value'])
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.90)
    .rename(columns={'value': 'global_p90'})
)

# 2) Filter to incident window (inclusive) in UTC+8
window_start_sh = pd.Timestamp('2021-03-25 02:30:00').tz_localize(sh)
window_end_sh = pd.Timestamp('2021-03-25 03:00:00').tz_localize(sh)
df_window = df_metric[(df_metric['ts_sh'] >= window_start_sh) & (df_metric['ts_sh'] <= window_end_sh)].copy()

# 3) Merge global_p90 and mark strict anomalies (value > global_p90)
df_win = df_window.merge(global_p90, on=['cmdb_id', 'kpi_name'], how='left')
df_win['is_anom_p90_strict'] = df_win['value'] > df_win['global_p90']

# Helper to compute longest consecutive run and other aggregates for a group
def compute_runs_metric(g):
    g = g.sort_values('timestamp').reset_index(drop=True)
    pts = int(g.shape[0])
    anomaly_count = int(g['is_anom_p90_strict'].sum())
    max_value = g['value'].max() if pts > 0 else None

    if anomaly_count == 0:
        return pd.Series({
            'points_in_window': pts,
            'anomaly_count_p90_strict': anomaly_count,
            'earliest_anomaly_timestamp_p90_strict': pd.NaT,
            'max_value_in_window': max_value,
            'longest_consecutive_anomalous_run': 0,
            'start_timestamp_of_longest_run': pd.NaT
        })

    longest_run = 0
    longest_start_ts = None
    current_run = 0
    current_start_ts = None

    for idx, row in g.iterrows():
        if row['is_anom_p90_strict']:
            if current_run == 0:
                current_run = 1
                current_start_ts = row['ts_sh']
            else:
                prev_row = g.iloc[idx-1]
                # require exact 60-second adjacency and previous point also anomalous
                if (row['timestamp'] - prev_row['timestamp']) == 60 and prev_row['is_anom_p90_strict']:
                    current_run += 1
                else:
                    current_run = 1
                    current_start_ts = row['ts_sh']
            if current_run > longest_run:
                longest_run = current_run
                longest_start_ts = current_start_ts
        else:
            current_run = 0
            current_start_ts = None

    # earliest anomaly timestamp (first > P90)
    earliest_idx = g[g['is_anom_p90_strict']].index[0]
    earliest_ts = g.loc[earliest_idx, 'ts_sh']

    return pd.Series({
        'points_in_window': pts,
        'anomaly_count_p90_strict': anomaly_count,
        'earliest_anomaly_timestamp_p90_strict': earliest_ts,
        'max_value_in_window': max_value,
        'longest_consecutive_anomalous_run': longest_run,
        'start_timestamp_of_longest_run': longest_start_ts
    })

# 4) Group and compute runs
runs_df = df_win.groupby(['cmdb_id', 'kpi_name']).apply(compute_runs_metric).reset_index()

# Merge global_p90 into runs_df for reporting
summary_df = runs_df.merge(global_p90, on=['cmdb_id', 'kpi_name'], how='left')

# Format timestamps to strings (UTC+8) or None for compact output
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(sh).strftime('%Y-%m-%d %H:%M:%S %Z')

summary_df['earliest_anomaly_timestamp_p90_strict'] = summary_df['earliest_anomaly_timestamp_p90_strict'].apply(fmt_ts)
summary_df['start_timestamp_of_longest_run'] = summary_df['start_timestamp_of_longest_run'].apply(fmt_ts)

# Ensure integer types
summary_df['points_in_window'] = summary_df['points_in_window'].astype(int)
summary_df['anomaly_count_p90_strict'] = summary_df['anomaly_count_p90_strict'].astype(int)
summary_df['longest_consecutive_anomalous_run'] = summary_df['longest_consecutive_anomalous_run'].astype(int)

# 5) Sort by longest run desc, then anomaly count desc. Get top 20.
summary_sorted = summary_df.sort_values(['longest_consecutive_anomalous_run', 'anomaly_count_p90_strict'], ascending=[False, False]).reset_index(drop=True)
top20_df = summary_sorted.head(20)[[
    'cmdb_id',
    'kpi_name',
    'global_p90',
    'points_in_window',
    'anomaly_count_p90_strict',
    'earliest_anomaly_timestamp_p90_strict',
    'max_value_in_window',
    'longest_consecutive_anomalous_run',
    'start_timestamp_of_longest_run'
]].copy()

# Candidate faults: longest run >= 3
candidates_df = summary_sorted[summary_sorted['longest_consecutive_anomalous_run'] >= 3].copy()
candidate_count = int(candidates_df.shape[0])
candidates_head20 = candidates_df.head(20)[[
    'cmdb_id',
    'kpi_name',
    'global_p90',
    'points_in_window',
    'anomaly_count_p90_strict',
    'earliest_anomaly_timestamp_p90_strict',
    'max_value_in_window',
    'longest_consecutive_anomalous_run',
    'start_timestamp_of_longest_run'
]]

# Prepare compact summary dictionary
result_summary = {
    "window_start": window_start_sh.strftime('%Y-%m-%d %H:%M:%S %Z'),
    "window_end": window_end_sh.strftime('%Y-%m-%d %H:%M:%S %Z'),
    "total_pairs_in_window": int(summary_df.shape[0]),
    "candidate_faults_count_longest_run_ge_3": candidate_count
}

# Display compact outputs: result summary, top20 DataFrame, and up-to-20 candidate pairs (if any)
result_summary, top20_df, candidates_head20

```
Out[6]:
```
Summary (plain English):

- Analysis: computed global P90 per (cmdb_id, kpi_name) from the full metric series, then flagged points in the incident window (2021-03-25 02:30:00 to 03:00:00 UTC+8) as anomalous if value > global P90 (strict).

- Scope: 1,764 distinct (cmdb_id, kpi_name) pairs were present in the window.

- Candidate faults: 19 pairs had a longest consecutive anomalous run >= 3 minutes (i.e., sustained >P90). These are the top candidates (cmdb_id / kpi_name — longest_run minutes — anomaly_count (>P90) — start of longest run):

  1. IG01 / JVM-Memory_7778_JVM_Memory_HeapMemoryUsed — 3 min — 3 anomalies — start 2021-03-25 02:58:00
  2. IG02 / JVM-Memory_7778_JVM_Memory_HeapMemoryUsed — 3 min — 3 anomalies — start 2021-03-25 02:58:00
  3. MG01 / JVM-Memory_7779_JVM_Memory_HeapMemoryUsed — 3 min — 3 anomalies — start 2021-03-25 02:49:00
  4. MG01 / OSLinux-OSLinux_ZABBIX_Host_Uptime — 3 min — 3 anomalies — start 2021-03-25 02:58:00
  5. MG02 / JVM-Memory_7779_JVM_Memory_HeapMemoryUsed — 3 min — 3 anomalies — start 2021-03-25 02:32:00
  6. MG02 / OSLinux-CPU_CPU_CPULoad — 3 min — 3 anomalies — start 2021-03-25 02:45:00
  7. MG02 / OSLinux-CPU_CPU_CPUSysTime — 3 min — 3 anomalies — start 2021-03-25 02:45:00
  8. MG02 / OSLinux-OSLinux_ZABBIX_Host_Uptime — 3 min — 3 anomalies — start 2021-03-25 02:58:00
  9. Mysql02 / Mysql-MySQL_3306_Sort Rows — 3 min — 3 anomalies — start 2021-03-25 02:57:00
 10. Tomcat01 / OSLinux-OSLinux_ZABBIX_Host_Uptime — 3 min — 3 anomalies — start 2021-03-25 02:58:00
 11. Tomcat01 / Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo — 3 min — 3 anomalies — start 2021-03-25 02:58:00
 12. Tomcat01 / Tomcat-Requests_7441-"http-nio-8003"_RequestCountRequestInfo — 3 min — 3 anomalies — start 2021-03-25 02:58:00
 13. Tomcat02 / OSLinux-OSLinux_ZABBIX_Host_Uptime — 3 min — 3 anomalies — start 2021-03-25 02:58:00
 14. Tomcat02 / Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo — 3 min — 3 anomalies — start 2021-03-25 02:58:00
 15. Tomcat02 / Tomcat-Requests_7441-"http-nio-8003"_RequestCountRequestInfo — 3 min — 3 anomalies — start 2021-03-25 02:58:00
 16. Tomcat03 / OSLinux-OSLinux_ZABBIX_Host_Uptime — 3 min — 3 anomalies — start 2021-03-25 02:58:00
 17. Tomcat03 / Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo — 3 min — 3 anomalies — start 2021-03-25 02:58:00
 18. Tomcat03 / Tomcat-Requests_7441-"http-nio-8003"_RequestCountRequestInfo — 3 min — 3 anomalies — start 2021-03-25 02:58:00
 19. Tomcat04 / OSLinux-OSLinux_ZABBIX_Host_Uptime — 3 min — 3 anomalies — start 2021-03-25 02:58:00

- Observations / interpretation:
  - Many candidate faults are JVM heap usage increases (IG/MG) and Tomcat request-processing/request-count spikes — all showing 3 consecutive minutes > their global P90, typically starting at or near 02:58 for Tomcat/IG/MG and at other times for some MG/Mysql events (e.g., MG02 heap at 02:32, MG01 heap at 02:49, Mysql02 sort rows at 02:57).
  - Several OSLinux ZABBIX host uptime KPIs are flagged (>P90) across Tomcat/MG hosts at 02:58 — likely reflecting a common synchronized metric behavior or clocked measurement bump; verify whether ZABBIX uptime is meaningful for anomaly detection.
  - Tomcat request processing time and request count anomalies coincide (same start times) across multiple Tomcat instances, suggesting system-wide request spike or measurement change around ~02:58.

- Next recommended actions:
  - Inspect traces and logs around the highlighted start timestamps (notably ~02:58, also ~02:32, ~02:45, ~02:49, ~02:57) for correlated errors or latency (trace.csv and log.csv findings).
  - Investigate JVM memory increases on IG/MG and Tomcat request spikes for root causes (traffic surge, GC events, downstream DB pressure).
  - Review whether ZABBIX/uptime metrics are noisy or should be excluded from fault lists.

If you want, I can export the 19 candidate pairs with their full per-minute window rows (compact) or cross-correlate these candidates with the trace/log anomalies at the same timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

({'window_start': '2021-03-25 02:30:00 CST', 'window_end': '2021-03-25 03:00:00 CST', 'total_pairs_in_window': 1764, 'candidate_faults_count_longest_run_ge_3': 19},      cmdb_id                                           kpi_name    global_p90  points_in_window  anomaly_count_p90_strict earliest_anomaly_timestamp_p90_strict  max_value_in_window  longest_consecutive_anomalous_run start_timestamp_of_longest_run
0       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  4.514665e+08                31                         3               2021-03-25 02:58:00 CST         4.612248e+08                                  3        2021-03-25 02:58:00 CST
1       IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  3.516385e+08                31                         3               2021-03-25 02:58:00 CST         3.597700e+08                                  3        2021-03-25 02:58:00 CST
2       MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  9.657777e+08                31                         3               2021-03-25 02:49:00 CST         1.023528e+09                                  3        2021-03-25 02:49:00 CST
3       MG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811922e+06                31                         3               2021-03-25 02:58:00 CST         7.812103e+06                                  3        2021-03-25 02:58:00 CST
4       MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  9.301490e+08                31                         3               2021-03-25 02:32:00 CST         9.803417e+08                                  3        2021-03-25 02:32:00 CST
5       MG02                            OSLinux-CPU_CPU_CPULoad  1.870000e+00                31                         3               2021-03-25 02:45:00 CST         2.160000e+00                                  3        2021-03-25 02:45:00 CST
6       MG02                         OSLinux-CPU_CPU_CPUSysTime  1.774700e+00                31                         3               2021-03-25 02:45:00 CST         2.023400e+00                                  3        2021-03-25 02:45:00 CST
7       MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811331e+06                31                         3               2021-03-25 02:58:00 CST         7.811511e+06                                  3        2021-03-25 02:58:00 CST
8    Mysql02                         Mysql-MySQL_3306_Sort Rows  1.213217e+06                29                         3               2021-03-25 02:57:00 CST         1.213281e+06                                  3        2021-03-25 02:57:00 CST
9   Tomcat01                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811221e+06                30                         3               2021-03-25 02:58:00 CST         7.811396e+06                                  3        2021-03-25 02:58:00 CST
10  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  2.123821e+09                31                         3               2021-03-25 02:58:00 CST         2.123821e+09                                  3        2021-03-25 02:58:00 CST
11  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  3.208037e+06                31                         3               2021-03-25 02:58:00 CST         3.208041e+06                                  3        2021-03-25 02:58:00 CST
12  Tomcat02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811154e+06                31                         3               2021-03-25 02:58:00 CST         7.811334e+06                                  3        2021-03-25 02:58:00 CST
13  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...  2.056740e+09                31                         3               2021-03-25 02:58:00 CST         2.056741e+09                                  3        2021-03-25 02:58:00 CST
14  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  3.221690e+06                31                         3               2021-03-25 02:58:00 CST         3.221694e+06                                  3        2021-03-25 02:58:00 CST
15  Tomcat03                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811139e+06                31                         3               2021-03-25 02:58:00 CST         7.811319e+06                                  3        2021-03-25 02:58:00 CST
16  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...  2.044341e+09                31                         3               2021-03-25 02:58:00 CST         2.044341e+09                                  3        2021-03-25 02:58:00 CST
17  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  3.208062e+06                31                         3               2021-03-25 02:58:00 CST         3.208067e+06                                  3        2021-03-25 02:58:00 CST
18  Tomcat04                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811120e+06                31                         3               2021-03-25 02:58:00 CST         7.811300e+06                                  3        2021-03-25 02:58:00 CST
19      MG01                            OSLinux-CPU_CPU_CPULoad  1.110000e+00                31                         3               2021-03-25 02:39:00 CST         1.420000e+00                                  2        2021-03-25 02:44:00 CST,      cmdb_id                                           kpi_name    global_p90  points_in_window  anomaly_count_p90_strict earliest_anomaly_timestamp_p90_strict  max_value_in_window  longest_consecutive_anomalous_run start_timestamp_of_longest_run
0       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  4.514665e+08                31                         3               2021-03-25 02:58:00 CST         4.612248e+08                                  3        2021-03-25 02:58:00 CST
1       IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  3.516385e+08                31                         3               2021-03-25 02:58:00 CST         3.597700e+08                                  3        2021-03-25 02:58:00 CST
2       MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  9.657777e+08                31                         3               2021-03-25 02:49:00 CST         1.023528e+09                                  3        2021-03-25 02:49:00 CST
3       MG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811922e+06                31                         3               2021-03-25 02:58:00 CST         7.812103e+06                                  3        2021-03-25 02:58:00 CST
4       MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  9.301490e+08                31                         3               2021-03-25 02:32:00 CST         9.803417e+08                                  3        2021-03-25 02:32:00 CST
5       MG02                            OSLinux-CPU_CPU_CPULoad  1.870000e+00                31                         3               2021-03-25 02:45:00 CST         2.160000e+00                                  3        2021-03-25 02:45:00 CST
6       MG02                         OSLinux-CPU_CPU_CPUSysTime  1.774700e+00                31                         3               2021-03-25 02:45:00 CST         2.023400e+00                                  3        2021-03-25 02:45:00 CST
7       MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811331e+06                31                         3               2021-03-25 02:58:00 CST         7.811511e+06                                  3        2021-03-25 02:58:00 CST
8    Mysql02                         Mysql-MySQL_3306_Sort Rows  1.213217e+06                29                         3               2021-03-25 02:57:00 CST         1.213281e+06                                  3        2021-03-25 02:57:00 CST
9   Tomcat01                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811221e+06                30                         3               2021-03-25 02:58:00 CST         7.811396e+06                                  3        2021-03-25 02:58:00 CST
10  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  2.123821e+09                31                         3               2021-03-25 02:58:00 CST         2.123821e+09                                  3        2021-03-25 02:58:00 CST
11  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  3.208037e+06                31                         3               2021-03-25 02:58:00 CST         3.208041e+06                                  3        2021-03-25 02:58:00 CST
12  Tomcat02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811154e+06                31                         3               2021-03-25 02:58:00 CST         7.811334e+06                                  3        2021-03-25 02:58:00 CST
13  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...  2.056740e+09                31                         3               2021-03-25 02:58:00 CST         2.056741e+09                                  3        2021-03-25 02:58:00 CST
14  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  3.221690e+06                31                         3               2021-03-25 02:58:00 CST         3.221694e+06                                  3        2021-03-25 02:58:00 CST
15  Tomcat03                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811139e+06                31                         3               2021-03-25 02:58:00 CST         7.811319e+06                                  3        2021-03-25 02:58:00 CST
16  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...  2.044341e+09                31                         3               2021-03-25 02:58:00 CST         2.044341e+09                                  3        2021-03-25 02:58:00 CST
17  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  3.208062e+06                31                         3               2021-03-25 02:58:00 CST         3.208067e+06                                  3        2021-03-25 02:58:00 CST
18  Tomcat04                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811120e+06                31                         3               2021-03-25 02:58:00 CST         7.811300e+06                                  3        2021-03-25 02:58:00 CST)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# reuse df_trace if available
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

sh = pytz.timezone('Asia/Shanghai')

# ensure numeric and timestamp conversions (reuse existing columns if present)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
if 'ts_utc' not in df_trace.columns:
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
if 'ts_sh' not in df_trace.columns:
    df_trace['ts_sh'] = df_trace['ts_utc'].dt.tz_convert(sh)

# 1) Compute global P90 per (cmdb_id, trace_name) using full trace series
trace_global_p90 = (
    df_trace
    .dropna(subset=['value'])
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.90)
    .rename(columns={'value': 'global_p90'})
)

# 2) Filter to timestamps between 2021-03-25 02:56:00 and 2021-03-25 03:00:00 (UTC+8)
window_start_sh = pd.Timestamp('2021-03-25 02:56:00').tz_localize(sh)
window_end_sh = pd.Timestamp('2021-03-25 03:00:00').tz_localize(sh)
df_window = df_trace[(df_trace['ts_sh'] >= window_start_sh) & (df_trace['ts_sh'] <= window_end_sh)].copy()

# 3) Restrict to candidate cmdb_ids
candidates = ['IG01','IG02','MG01','MG02','Mysql02','Tomcat01','Tomcat02','Tomcat03','Tomcat04']
df_window = df_window[df_window['cmdb_id'].isin(candidates)].copy()

# 4) Merge global_p90 and aggregate per (cmdb_id, trace_name)
df_win = df_window.merge(trace_global_p90, on=['cmdb_id','trace_name'], how='left')

# mark above-p90 (strict)
df_win['above_p90'] = df_win['value'] > df_win['global_p90']

# Aggregations
agg = (
    df_win
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(
        points_in_window=('value','size'),
        max_value_in_window=('value','max'),
        mean_value_in_window=('value','mean'),
        count_above_p90=('above_p90','sum')
    )
)

# earliest timestamp above p90 (per group)
earliest = (
    df_win[df_win['above_p90']]
    .groupby(['cmdb_id','trace_name'], as_index=False)['ts_sh']
    .min()
    .rename(columns={'ts_sh':'earliest_timestamp_above_p90'})
)

# combine
result = agg.merge(trace_global_p90, on=['cmdb_id','trace_name'], how='left').merge(earliest, on=['cmdb_id','trace_name'], how='left')

# format earliest timestamp strings or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(sh).strftime('%Y-%m-%d %H:%M:%S %Z')

result['earliest_timestamp_above_p90'] = result['earliest_timestamp_above_p90'].apply(fmt_ts)

# tidy numeric columns: round mean for compactness
result['mean_value_in_window'] = result['mean_value_in_window'].round(6)

# 5) Sort by count_above_p90 desc, then max_value_in_window desc; return top 20
result_sorted = result.sort_values(['count_above_p90','max_value_in_window'], ascending=[False, False]).reset_index(drop=True)
top20 = result_sorted.head(20)[[
    'cmdb_id','trace_name','global_p90','points_in_window','max_value_in_window','mean_value_in_window',
    'count_above_p90','earliest_timestamp_above_p90'
]]

# short list where count_above_p90 >= 2 (concentrated deviation at/around 02:58)
concentrated = result_sorted[result_sorted['count_above_p90'] >= 2].head(20)[[
    'cmdb_id','trace_name','global_p90','points_in_window','max_value_in_window','mean_value_in_window',
    'count_above_p90','earliest_timestamp_above_p90'
]]

# Compact summary meta
summary = {
    "window_start": window_start_sh.strftime('%Y-%m-%d %H:%M:%S %Z'),
    "window_end": window_end_sh.strftime('%Y-%m-%d %H:%M:%S %Z'),
    "rows_in_window_after_filter": int(df_window.shape[0]),
    "distinct_pairs_in_window": int(result.shape[0]),
    "pairs_with_count_above_p90_ge_1": int((result['count_above_p90'] > 0).sum()),
    "pairs_with_count_above_p90_ge_2": int((result['count_above_p90'] >= 2).sum())
}

# Display compact outputs
summary, top20, concentrated

```
Out[7]:
```
```python
# Plain-English summary of the trace-focused scan (02:56–03:00 UTC+8) — concise and JSON-friendly
summary = (
    "Trace scan (2021-03-25 02:56:00 to 03:00:00 UTC+8):\n"
    "- Rows after filtering: 225; distinct (cmdb_id,trace_name) pairs: 111.\n"
    "- P90-based deviations: 53 pairs had at least one value > global P90; 20 pairs had >=2 values > P90 (concentrated deviations).\n\n"
    "Key observations:\n"
    "- The strongest short-lived spike is Tomcat01 / trace.self.duration_mean: 3 points in-window, all 3 > P90, earliest >P90 at 02:57:00.\n"
    "- Many concentrated deviations (count_above_p90 = 2) cluster at ~02:57–02:58 across IG, MG, Tomcat paths. Examples include:\n"
    "  * IG01 → trace.to_Tomcat02.duration_mean & duration_p95 (2 points, earliest 02:57)\n"
    "  * Tomcat02 ← trace.from_IG01.duration_mean & duration_p95 (2 points, earliest 02:57)\n"
    "  * MG02 ←/→ trace.from/to Tomcat02 (duration_mean/duration_p95) (2 points, earliest 02:57)\n"
    "  * IG02 and Tomcat01/Tomcat03 trace.self.* and trace.to/from pairs show 2-point deviations with earliest timestamps at 02:58:00.\n\n"
    "Interpretation / next steps:\n"
    "- There is a short, concentrated deviation event centered around 02:57–02:58 affecting request/latency traces between IG ↔ Tomcat ↔ MG and some Tomcat self-duration metrics.\n"
    "- Recommended follow-ups: inspect traces and logs at 02:56–02:59 for those components (IG01/IG02, MG02, Tomcat01–03, Mysql02) to find correlated latency spikes, request surges, or downstream bottlenecks.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

({'window_start': '2021-03-25 02:56:00 CST', 'window_end': '2021-03-25 03:00:00 CST', 'rows_in_window_after_filter': 225, 'distinct_pairs_in_window': 111, 'pairs_with_count_above_p90_ge_1': 53, 'pairs_with_count_above_p90_ge_2': 20},      cmdb_id                         trace_name  global_p90  points_in_window  max_value_in_window  mean_value_in_window  count_above_p90 earliest_timestamp_above_p90
0   Tomcat01           trace.self.duration_mean    0.010358                 3             0.014917              0.013639                3      2021-03-25 02:57:00 CST
1       IG01    trace.to_Tomcat02.duration_mean    0.161500                 2             0.283000              0.232500                2      2021-03-25 02:57:00 CST
2       IG01     trace.to_Tomcat02.duration_p95    0.251810                 2             0.283000              0.281100                2      2021-03-25 02:57:00 CST
3   Tomcat02      trace.from_IG01.duration_mean    0.161500                 2             0.283000              0.232500                2      2021-03-25 02:57:00 CST
4   Tomcat02       trace.from_IG01.duration_p95    0.251810                 2             0.283000              0.281100                2      2021-03-25 02:57:00 CST
5       MG02  trace.from_Tomcat02.duration_mean    0.163350                 2             0.276000              0.225750                2      2021-03-25 02:57:00 CST
6       MG02   trace.from_Tomcat02.duration_p95    0.260345                 2             0.276000              0.274125                2      2021-03-25 02:57:00 CST
7   Tomcat02        trace.to_MG02.duration_mean    0.163350                 2             0.276000              0.225750                2      2021-03-25 02:57:00 CST
8   Tomcat02         trace.to_MG02.duration_p95    0.260345                 2             0.276000              0.274125                2      2021-03-25 02:57:00 CST
9       MG02     trace.to_dockerB2.duration_p95    0.204435                 2             0.272000              0.270225                2      2021-03-25 02:57:00 CST
10  Tomcat02            trace.self.duration_p95    0.074470                 2             0.209250              0.169350                2      2021-03-25 02:57:00 CST
11      IG02           trace.self.duration_mean    0.083300                 3             0.111000              0.093167                2      2021-03-25 02:58:00 CST
12      IG02    trace.to_Tomcat01.duration_mean    0.080333                 3             0.093500              0.085833                2      2021-03-25 02:58:00 CST
13  Tomcat01      trace.from_IG02.duration_mean    0.080333                 3             0.093500              0.085833                2      2021-03-25 02:58:00 CST
14  Tomcat01            trace.self.duration_p95    0.058500                 3             0.087650              0.068217                2      2021-03-25 02:58:00 CST
15      MG02  trace.from_Tomcat01.duration_mean    0.050933                 2             0.087500              0.079250                2      2021-03-25 02:57:00 CST
16  Tomcat01        trace.to_MG02.duration_mean    0.050933                 2             0.087500              0.079250                2      2021-03-25 02:57:00 CST
17  Tomcat03            trace.self.duration_p95    0.051800                 2             0.079100              0.066925                2      2021-03-25 02:58:00 CST
18  Tomcat02           trace.self.duration_mean    0.023400                 2             0.046500              0.035944                2      2021-03-25 02:57:00 CST
19  Tomcat03           trace.self.duration_mean    0.011789                 2             0.016542              0.014521                2      2021-03-25 02:58:00 CST,      cmdb_id                         trace_name  global_p90  points_in_window  max_value_in_window  mean_value_in_window  count_above_p90 earliest_timestamp_above_p90
0   Tomcat01           trace.self.duration_mean    0.010358                 3             0.014917              0.013639                3      2021-03-25 02:57:00 CST
1       IG01    trace.to_Tomcat02.duration_mean    0.161500                 2             0.283000              0.232500                2      2021-03-25 02:57:00 CST
2       IG01     trace.to_Tomcat02.duration_p95    0.251810                 2             0.283000              0.281100                2      2021-03-25 02:57:00 CST
3   Tomcat02      trace.from_IG01.duration_mean    0.161500                 2             0.283000              0.232500                2      2021-03-25 02:57:00 CST
4   Tomcat02       trace.from_IG01.duration_p95    0.251810                 2             0.283000              0.281100                2      2021-03-25 02:57:00 CST
5       MG02  trace.from_Tomcat02.duration_mean    0.163350                 2             0.276000              0.225750                2      2021-03-25 02:57:00 CST
6       MG02   trace.from_Tomcat02.duration_p95    0.260345                 2             0.276000              0.274125                2      2021-03-25 02:57:00 CST
7   Tomcat02        trace.to_MG02.duration_mean    0.163350                 2             0.276000              0.225750                2      2021-03-25 02:57:00 CST
8   Tomcat02         trace.to_MG02.duration_p95    0.260345                 2             0.276000              0.274125                2      2021-03-25 02:57:00 CST
9       MG02     trace.to_dockerB2.duration_p95    0.204435                 2             0.272000              0.270225                2      2021-03-25 02:57:00 CST
10  Tomcat02            trace.self.duration_p95    0.074470                 2             0.209250              0.169350                2      2021-03-25 02:57:00 CST
11      IG02           trace.self.duration_mean    0.083300                 3             0.111000              0.093167                2      2021-03-25 02:58:00 CST
12      IG02    trace.to_Tomcat01.duration_mean    0.080333                 3             0.093500              0.085833                2      2021-03-25 02:58:00 CST
13  Tomcat01      trace.from_IG02.duration_mean    0.080333                 3             0.093500              0.085833                2      2021-03-25 02:58:00 CST
14  Tomcat01            trace.self.duration_p95    0.058500                 3             0.087650              0.068217                2      2021-03-25 02:58:00 CST
15      MG02  trace.from_Tomcat01.duration_mean    0.050933                 2             0.087500              0.079250                2      2021-03-25 02:57:00 CST
16  Tomcat01        trace.to_MG02.duration_mean    0.050933                 2             0.087500              0.079250                2      2021-03-25 02:57:00 CST
17  Tomcat03            trace.self.duration_p95    0.051800                 2             0.079100              0.066925                2      2021-03-25 02:58:00 CST
18  Tomcat02           trace.self.duration_mean    0.023400                 2             0.046500              0.035944                2      2021-03-25 02:57:00 CST
19  Tomcat03           trace.self.duration_mean    0.011789                 2             0.016542              0.014521                2      2021-03-25 02:58:00 CST)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

sh = pytz.timezone('Asia/Shanghai')

# Ensure numeric and timestamp columns exist (reuse if already computed)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'ts_utc' not in df_metric.columns:
    df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
if 'ts_sh' not in df_metric.columns:
    df_metric['ts_sh'] = df_metric['ts_utc'].dt.tz_convert(sh)

# 1) Compute global P90 per (cmdb_id, kpi_name) using full series (do NOT filter first)
try:
    global_p90  # reuse if exists
except NameError:
    global_p90 = (
        df_metric
        .dropna(subset=['value'])
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.90)
        .rename(columns={'value': 'global_p90'})
    )

# 2) Restrict to Tomcat cmdb_ids and selected KPI name substrings
tomcats = ['Tomcat01', 'Tomcat02', 'Tomcat03']
substrs = ['Tomcat-Requests', 'JVM-Memory', 'OSLinux-OSLinux_ZABBIX_Host_Uptime']
pattern = '|'.join([s.replace('"','') for s in substrs])  # basic pattern

df_focus = df_metric[
    (df_metric['cmdb_id'].isin(tomcats)) &
    (df_metric['kpi_name'].str.contains(pattern, na=False))
].copy()

# 3) Filter to timestamps between 2021-03-25 02:56:00 and 2021-03-25 03:00:00 UTC+8 (inclusive)
window_start_sh = pd.Timestamp('2021-03-25 02:56:00').tz_localize(sh)
window_end_sh = pd.Timestamp('2021-03-25 03:00:00').tz_localize(sh)
df_window = df_focus[(df_focus['ts_sh'] >= window_start_sh) & (df_focus['ts_sh'] <= window_end_sh)].copy()

# If no rows, produce empty summary
if df_window.empty:
    top20_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p90','points_in_window','count_above_p90',
        'earliest_timestamp_above_p90','max_value_in_window','sample_rows_recent_first'
    ])
    total_pairs = 0
else:
    # 4) Merge global_p90 (precomputed) and mark > global_p90 (strict)
    df_win = df_window.merge(global_p90, on=['cmdb_id','kpi_name'], how='left')
    df_win['above_p90'] = df_win['value'] > df_win['global_p90']

    # Aggregations per pair
    agg = (
        df_win
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(
            points_in_window=('value','size'),
            count_above_p90=('above_p90','sum'),
            max_value_in_window=('value','max')
        )
    )

    # earliest timestamp above p90 per group
    earliest = (
        df_win[df_win['above_p90']]
        .groupby(['cmdb_id','kpi_name'], as_index=False)['ts_sh']
        .min()
        .rename(columns={'ts_sh':'earliest_ts_above_p90'})
    )

    # merge
    summary = agg.merge(global_p90, on=['cmdb_id','kpi_name'], how='left').merge(earliest, on=['cmdb_id','kpi_name'], how='left')

    # format earliest timestamp string or None
    summary['earliest_timestamp_above_p90'] = summary['earliest_ts_above_p90'].apply(
        lambda x: x.tz_convert(sh).strftime('%Y-%m-%d %H:%M:%S %Z') if pd.notna(x) else None
    )

    # 4b) collect up to 10 recent rows per pair (most recent first)
    # we'll build a samples dict and then map to summary
    samples = {}
    grouped = df_win.sort_values('timestamp', ascending=False).groupby(['cmdb_id','kpi_name'])
    for (cm, kp), g in grouped:
        rows = g[['ts_sh','value']].head(10).copy()
        rows['ts_str'] = rows['ts_sh'].dt.tz_convert(sh).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        samples[(cm,kp)] = list(rows[['ts_str','value']].itertuples(index=False, name=None))

    # attach samples as small lists
    summary['sample_rows_recent_first'] = summary.apply(lambda r: samples.get((r['cmdb_id'], r['kpi_name']), []), axis=1)

    # reorder/select columns for compact output
    top_df = summary[[
        'cmdb_id','kpi_name','global_p90','points_in_window','count_above_p90',
        'earliest_timestamp_above_p90','max_value_in_window','sample_rows_recent_first'
    ]].copy()

    # 5) Sort by count_above_p90 desc then max_value_in_window desc, take top 20
    top20_df = top_df.sort_values(['count_above_p90','max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

    total_pairs = int(top_df.shape[0])

# Final compact outputs (DataFrame and total pair count)
top20_df, total_pairs

```
Out[8]:
```
Summary (plain English):

- Scope: metric data for Tomcat01/02/03 during 2021-03-25 02:56:00–03:00:00 (UTC+8). Global P90 thresholds were computed from the full metric series and points were flagged when value > global P90.

- Key confirmed findings (top candidates, each with points_in_window = 5 and count_above_p90 = 3):
  1. Tomcat01 — Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo
     - global_p90 ≈ 2.123821e9, 5 points in window, 3 points > P90
     - earliest >P90 at 2021-03-25 02:58:00; max ≈ 2,123,821,228
     - sample recent values include 2021-03-25 03:00:00 -> 2,123,821,228
  2. Tomcat02 — same ProcessingTimeRequestInfo
     - global_p90 ≈ 2.056740e9, 5 pts, 3 > P90, earliest 02:58:00; max ≈ 2,056,740,859
  3. Tomcat03 — same ProcessingTimeRequestInfo
     - global_p90 ≈ 2.044341e9, 5 pts, 3 > P90, earliest 02:58:00; max ≈ 2,044,341,420
  4. Tomcat01 — OSLinux-OSLinux_ZABBIX_Host_Uptime
     - global_p90 ≈ 7,811,221; 5 pts, 3 > P90; earliest 02:58:00; max ≈ 7,811,396
  5. Tomcat02 — OSLinux-OSLinux_ZABBIX_Host_Uptime (same pattern)
  6. Tomcat03 — OSLinux-OSLinux_ZABBIX_Host_Uptime (same pattern)
  7. Tomcat02 — Tomcat-Requests_..._RequestCountRequestInfo
     - global_p90 ≈ 3.221690e6, 5 pts, 3 > P90; earliest 02:58:00; max ≈ 3,221,694
  8. Tomcat03 — RequestCount (similar)
  9. Tomcat01 — RequestCount (similar)

- Metrics that did NOT show >P90 in this window (examples): Request MaxTime and ErrorCount for these Tomcats had 0 count_above_p90 in the window.

- Interpretation:
  - There is a clear, short-lived spike concentrated around 02:58 affecting Tomcat request processing time and request counts across Tomcat01/02/03 (each with 3 measurements > their global P90 within the 5-minute window). Zabbix uptime metrics also exceed P90 at the same timestamps (likely a synchronized measurement bump).
  - The presence of elevated ProcessingTime and RequestCount together across multiple Tomcat instances at the same time suggests a system-wide request/latency event (traffic surge or downstream slowdown) rather than an isolated instance issue.

- Recommended next steps:
  1. Inspect Tomcat logs and access traces at ~02:56–03:00 (especially 02:57–02:58) for error traces, slow requests, or client surges.
  2. Check JVM/GC logs and heap usage around the same times for memory pressure or long GC pauses.
  3. Correlate with trace.csv anomalies (IG ↔ Tomcat ↔ MG paths) and DB metrics (e.g., Mysql02 candidates) to find downstream bottlenecks.
  4. Ignore or verify ZABBIX uptime metric behavior (may be noisy or not meaningful as a fault indicator).

If you want, I can expand any candidate pair into a compact time-series table (per-minute rows) or cross-correlate these Tomcat candidates with the trace and DB metric candidates around 02:56–02:59.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name    global_p90  points_in_window  count_above_p90 earliest_timestamp_above_p90  max_value_in_window                           sample_rows_recent_first
0   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  2.123821e+09                 5                3      2021-03-25 02:58:00 CST         2.123821e+09  [(2021-03-25 03:00:00 CST, 2123821228.0), (202...
1   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...  2.056740e+09                 5                3      2021-03-25 02:58:00 CST         2.056741e+09  [(2021-03-25 03:00:00 CST, 2056740859.0), (202...
2   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...  2.044341e+09                 5                3      2021-03-25 02:58:00 CST         2.044341e+09  [(2021-03-25 03:00:00 CST, 2044341420.0), (202...
3   Tomcat01                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811221e+06                 5                3      2021-03-25 02:58:00 CST         7.811396e+06  [(2021-03-25 03:00:00 CST, 7811396.0), (2021-0...
4   Tomcat02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811154e+06                 5                3      2021-03-25 02:58:00 CST         7.811334e+06  [(2021-03-25 03:00:00 CST, 7811334.0), (2021-0...
5   Tomcat03                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.811139e+06                 5                3      2021-03-25 02:58:00 CST         7.811319e+06  [(2021-03-25 03:00:00 CST, 7811319.0), (2021-0...
6   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  3.221690e+06                 5                3      2021-03-25 02:58:00 CST         3.221694e+06  [(2021-03-25 03:00:00 CST, 3221694.0), (2021-0...
7   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  3.208062e+06                 5                3      2021-03-25 02:58:00 CST         3.208067e+06  [(2021-03-25 03:00:00 CST, 3208067.0), (2021-0...
8   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  3.208037e+06                 5                3      2021-03-25 02:58:00 CST         3.208041e+06  [(2021-03-25 03:00:00 CST, 3208041.0), (2021-0...
9   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  4.016560e+05                 5                0                         None         4.016560e+05  [(2021-03-25 03:00:00 CST, 401656.0), (2021-03...
10  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  4.002060e+05                 5                0                         None         4.002060e+05  [(2021-03-25 03:00:00 CST, 400206.0), (2021-03...
11  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.978080e+05                 5                0                         None         3.978080e+05  [(2021-03-25 03:00:00 CST, 397808.0), (2021-03...
12  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  1.093000e+03                 5                0                         None         1.093000e+03  [(2021-03-25 03:00:00 CST, 1093.0), (2021-03-2...
13  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  1.051000e+03                 5                0                         None         1.051000e+03  [(2021-03-25 03:00:00 CST, 1051.0), (2021-03-2...
14  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  9.940000e+02                 5                0                         None         9.940000e+02  [(2021-03-25 03:00:00 CST, 994.0), (2021-03-25..., 15)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# Reuse previously loaded dataframes if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Timezone
sh = pytz.timezone('Asia/Shanghai')

# Ensure numeric values and timestamp columns
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

for df in [(df_metric, 'metric'), (df_log, 'log')]:
    d = df[0]
    if 'ts_utc' not in d.columns:
        d['ts_utc'] = pd.to_datetime(d['timestamp'], unit='s', utc=True)
    if 'ts_sh' not in d.columns:
        d['ts_sh'] = d['ts_utc'].dt.tz_convert(sh)

# 1) Identify Tomcat01 KPIs and logs of interest
cm = 'Tomcat01'
metric_substrs = ['Tomcat-Requests', 'JVM-Memory', 'OSLinux-CPU']
# filter metric rows for Tomcat01 matching substrings (case-sensitive matching as data appears)
mask_metric_select = (df_metric['cmdb_id'] == cm) & df_metric['kpi_name'].str.contains('|'.join(metric_substrs), na=False)
df_metric_tomcat = df_metric[mask_metric_select].copy()

# log names of interest for Tomcat01
log_names = ['log.gc_pause_max', 'log.error_count']
df_log_tomcat = df_log[(df_log['cmdb_id'] == cm) & df_log['log_name'].isin(log_names)].copy()

# 2) Compute global P90 for each selected (cmdb_id, kpi_name) and (cmdb_id, log_name) using full series
# metric global p90 (for all pairs, but we'll merge only for Tomcat01 later)
metric_p90_all = (
    df_metric
    .dropna(subset=['value'])
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .quantile(0.90)
    .rename(columns={'value':'global_p90'})
)

# log global p90
log_p90_all = (
    df_log
    .dropna(subset=['value'])
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .quantile(0.90)
    .rename(columns={'value':'global_p90'})
)

# 3) Filter to incident sub-window 2021-03-25 02:56:00 - 2021-03-25 03:00:00 (UTC+8)
window_start = pd.Timestamp('2021-03-25 02:56:00').tz_localize(sh)
window_end = pd.Timestamp('2021-03-25 03:00:00').tz_localize(sh)

df_metric_win = df_metric_tomcat[(df_metric_tomcat['ts_sh'] >= window_start) & (df_metric_tomcat['ts_sh'] <= window_end)].copy()
df_log_win = df_log_tomcat[(df_log_tomcat['ts_sh'] >= window_start) & (df_log_tomcat['ts_sh'] <= window_end)].copy()

# Helper to aggregate series info and sample rows
def summarize_metric_group(df_win, p90_df):
    if df_win.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','name','type','global_p90','points_in_window','count_above_p90',
            'earliest_timestamp_above_p90','max_value_in_window','sample_rows_recent_first'
        ])
    # merge p90
    merged = df_win.merge(p90_df, left_on=['cmdb_id','kpi_name'], right_on=['cmdb_id','kpi_name'], how='left')
    # mark above p90 (strict)
    merged['above_p90'] = merged['value'] > merged['global_p90']
    # aggregate
    agg = merged.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        points_in_window=('value','size'),
        count_above_p90=('above_p90','sum'),
        max_value_in_window=('value','max')
    )
    # earliest timestamp above p90
    earliest = merged[merged['above_p90']].groupby(['cmdb_id','kpi_name'], as_index=False)['ts_sh'].min().rename(columns={'ts_sh':'earliest_ts'})
    agg = agg.merge(earliest, on=['cmdb_id','kpi_name'], how='left')
    # attach global_p90
    agg = agg.merge(p90_df, on=['cmdb_id','kpi_name'], how='left')
    # samples: up to 10 most recent rows per group with flag
    samples = {}
    grp = merged.sort_values('timestamp', ascending=False).groupby(['cmdb_id','kpi_name'])
    for (cmdb,kpi), g in grp:
        rows = g[['ts_sh','value','above_p90']].head(10).copy()
        rows['ts_str'] = rows['ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        samples[(cmdb,kpi)] = list(rows[['ts_str','value','above_p90']].itertuples(index=False, name=None))
    agg['sample_rows_recent_first'] = agg.apply(lambda r: samples.get((r['cmdb_id'], r['kpi_name']), []), axis=1)
    # format earliest ts
    agg['earliest_timestamp_above_p90'] = agg['earliest_ts'].apply(lambda x: x.strftime('%Y-%m-%d %H:%M:%S %Z') if pd.notna(x) else None)
    # rename columns for uniform output
    agg = agg.rename(columns={'kpi_name':'name'})
    agg['type'] = 'metric'
    # select and order columns
    agg = agg[[
        'cmdb_id','name','type','global_p90','points_in_window','count_above_p90',
        'earliest_timestamp_above_p90','max_value_in_window','sample_rows_recent_first'
    ]]
    return agg

def summarize_log_group(df_win_log, p90_df_log):
    if df_win_log.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','name','type','global_p90','points_in_window','count_above_p90',
            'earliest_timestamp_above_p90','max_value_in_window','sample_rows_recent_first'
        ])
    merged = df_win_log.merge(p90_df_log, left_on=['cmdb_id','log_name'], right_on=['cmdb_id','log_name'], how='left')
    merged['above_p90'] = merged['value'] > merged['global_p90']
    agg = merged.groupby(['cmdb_id','log_name'], as_index=False).agg(
        points_in_window=('value','size'),
        count_above_p90=('above_p90','sum'),
        max_value_in_window=('value','max')
    )
    earliest = merged[merged['above_p90']].groupby(['cmdb_id','log_name'], as_index=False)['ts_sh'].min().rename(columns={'ts_sh':'earliest_ts'})
    agg = agg.merge(earliest, on=['cmdb_id','log_name'], how='left')
    agg = agg.merge(p90_df_log, on=['cmdb_id','log_name'], how='left')
    samples = {}
    grp = merged.sort_values('timestamp', ascending=False).groupby(['cmdb_id','log_name'])
    for (cmdb,ln), g in grp:
        rows = g[['ts_sh','value','above_p90']].head(10).copy()
        rows['ts_str'] = rows['ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        samples[(cmdb,ln)] = list(rows[['ts_str','value','above_p90']].itertuples(index=False, name=None))
    agg['sample_rows_recent_first'] = agg.apply(lambda r: samples.get((r['cmdb_id'], r['log_name']), []), axis=1)
    agg['earliest_timestamp_above_p90'] = agg['earliest_ts'].apply(lambda x: x.strftime('%Y-%m-%d %H:%M:%S %Z') if pd.notna(x) else None)
    agg = agg.rename(columns={'log_name':'name'})
    agg['type'] = 'log'
    agg = agg[[
        'cmdb_id','name','type','global_p90','points_in_window','count_above_p90',
        'earliest_timestamp_above_p90','max_value_in_window','sample_rows_recent_first'
    ]]
    return agg

# 4) Summarize metric and log groups
metrics_summary = summarize_metric_group(df_metric_win, metric_p90_all)
logs_summary = summarize_log_group(df_log_win, log_p90_all)

# Combine and sort by count_above_p90 desc then max_value desc
combined = pd.concat([metrics_summary, logs_summary], ignore_index=True)
if not combined.empty:
    combined_sorted = combined.sort_values(['count_above_p90','max_value_in_window'], ascending=[False, False]).reset_index(drop=True)
else:
    combined_sorted = combined

# Limit to Tomcat01 KPIs only (already filtered) and max 20 rows
result_df = combined_sorted.head(20).copy()

# Keep output compact: show selected columns and small samples
result_df[['cmdb_id','name','type','global_p90','points_in_window','count_above_p90',
           'earliest_timestamp_above_p90','max_value_in_window','sample_rows_recent_first']], result_df.shape[0]

```
Out[9]:
```
```python
# Plain-English summary for Tomcat01 (metrics + logs) — compact
summary = (
    "Tomcat01 focused summary (2021-03-25 02:56:00–03:00:00 UTC+8):\n\n"
    "- Key anomalies:\n"
    "  * Tomcat-Requests (ProcessingTimeRequestInfo): 5 points in window, 3 values > global P90, earliest >P90 at 2021-03-25 02:58:00, max ≈ 2.123821228e9.\n"
    "  * Tomcat-Requests (RequestCountRequestInfo): 5 points, 3 > P90, earliest >P90 at 2021-03-25 02:58:00, max ≈ 3.208041e6.\n"
    "  * OSLinux-CPU_CPU_CPULoad: 5 points, 2 values > P90, earliest >P90 at 2021-03-25 02:56:00, max ≈ 1.16.\n\n"
    "- Other checks:\n"
    "  * Tomcat MaxTime and ErrorCount metrics in this window did NOT exceed their global P90 (0 count_above_p90).\n"
    "  * log.error_count had 3 log points in-window but 0 > global P90. No log.gc_pause_max entries exceeded P90 in the window (no gc-induced spike observed).\n\n"
    "- Interpretation:\n"
    "  The Tomcat request processing time and request count spikes are synchronized (earliest strict exceedances at ~02:58) and present across Tomcat01 (also seen across Tomcat02/03). This pattern suggests a system-wide request/latency event (traffic surge or downstream slowdown) rather than an isolated Tomcat JVM failure.\n\n"
    "- Recommended next steps:\n"
    "  1) Inspect Tomcat access/error logs around 02:56–03:00 for slow requests, stack traces, or client bursts.\n  2) Check JVM GC/heap logs around 02:57–02:59 for pauses or memory pressure (even though gc_pause_max did not spike in the normalized logs here).\n  3) Correlate with trace and DB (Mysql02) anomalies at ~02:57–02:58 to identify downstream bottlenecks.\n\n"
    "If you want, I can return the sampled per-minute rows shown for each metric (they were collected most-recent-first) for deeper inspection."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                               name    type    global_p90  points_in_window  count_above_p90 earliest_timestamp_above_p90  max_value_in_window                           sample_rows_recent_first
0   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  metric  2.123821e+09                 5                3      2021-03-25 02:58:00 CST         2.123821e+09  [(2021-03-25 03:00:00 CST, 2123821228.0, True)...
1   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  metric  3.208037e+06                 5                3      2021-03-25 02:58:00 CST         3.208041e+06  [(2021-03-25 03:00:00 CST, 3208041.0, True), (...
2   Tomcat01                            OSLinux-CPU_CPU_CPULoad  metric  1.052000e+00                 5                2      2021-03-25 02:56:00 CST         1.160000e+00  [(2021-03-25 03:00:00 CST, 1.0, False), (2021-...
3   Tomcat01                             OSLinux-CPU_CPU_CPUWio  metric  1.297138e-01                 5                1      2021-03-25 02:57:00 CST         2.286140e-01  [(2021-03-25 03:00:00 CST, 0.01663, False), (2...
4   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  metric  3.978080e+05                 5                0                         None         3.978080e+05  [(2021-03-25 03:00:00 CST, 397808.0, False), (...
5   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  metric  1.051000e+03                 5                0                         None         1.051000e+03  [(2021-03-25 03:00:00 CST, 1051.0, False), (20...
6   Tomcat01                    OSLinux-CPU_CPU-2_SingleCpuUtil  metric  1.000000e+02                 1                0                         None         1.000000e+02          [(2021-03-25 02:59:00 CST, 100.0, False)]
7   Tomcat01                    OSLinux-CPU_CPU-3_SingleCpuidle  metric  9.958424e+01                 2                0                         None         9.950067e+01  [(2021-03-25 02:59:00 CST, 99.3348, False), (2...
8   Tomcat01                    OSLinux-CPU_CPU-1_SingleCpuidle  metric  9.958755e+01                 2                0                         None         9.938497e+01  [(2021-03-25 02:59:00 CST, 99.368, False), (20...
9   Tomcat01                    OSLinux-CPU_CPU-0_SingleCpuidle  metric  9.963428e+01                 2                0                         None         9.890290e+01  [(2021-03-25 02:59:00 CST, 98.9029, False), (2...
10  Tomcat01                        OSLinux-CPU_CPU_CPUidleutil  metric  7.464171e+01                 5                0                         None         7.456359e+01  [(2021-03-25 03:00:00 CST, 74.50214100000001, ...
11  Tomcat01                         OSLinux-CPU_CPU_CPUCpuUtil  metric  2.562250e+01                 4                0                         None         2.561730e+01  [(2021-03-25 03:00:00 CST, 25.4979, False), (2...
12  Tomcat01                        OSLinux-CPU_CPU_CPUUserTime  metric  2.512168e+01                 5                0                         None         2.510705e+01  [(2021-03-25 03:00:00 CST, 25.107055, False), ...
13  Tomcat01                    OSLinux-CPU_CPU-0_SingleCpuUtil  metric  1.770111e+01                 1                0                         None         1.313200e+00         [(2021-03-25 02:59:00 CST, 1.3132, False)]
14  Tomcat01                    OSLinux-CPU_CPU-3_SingleCpuUtil  metric  1.098200e+00                 2                0                         None         6.652000e-01  [(2021-03-25 03:00:00 CST, 0.6652, False), (20...
15  Tomcat01                    OSLinux-CPU_CPU-1_SingleCpuUtil  metric  1.000000e+02                 2                0                         None         6.320000e-01  [(2021-03-25 03:00:00 CST, 0.632, False), (202...
16  Tomcat01                         OSLinux-CPU_CPU_CPUSysTime  metric  2.997200e-01                 5                0                         None         2.993000e-01  [(2021-03-25 03:00:00 CST, 0.2494489999999999,...
17  Tomcat01                    OSLinux-CPU_CPU-2_SingleCpuidle  metric  9.973063e+01                 2                0                         None         0.000000e+00  [(2021-03-25 02:59:00 CST, 0.0, False), (2021-...
18  Tomcat01                                    log.error_count     log  0.000000e+00                 3                0                         None         0.000000e+00  [(2021-03-25 02:59:00 CST, 0.0, False), (2021-..., 19)```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

# Reuse df_metric if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

sh = pytz.timezone('Asia/Shanghai')

# Ensure timestamp and numeric columns exist
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'ts_utc' not in df_metric.columns:
    df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
if 'ts_sh' not in df_metric.columns:
    df_metric['ts_sh'] = df_metric['ts_utc'].dt.tz_convert(sh)

# 1) Restrict to Mysql02 and list available kpi_name values
mysql_cm = 'Mysql02'
mysql_df_all = df_metric[df_metric['cmdb_id'] == mysql_cm].copy()
mysql_kpis = sorted(mysql_df_all['kpi_name'].dropna().unique().tolist())
mysql_kpi_df = pd.DataFrame({'kpi_name': mysql_kpis})
mysql_kpi_count = mysql_kpi_df.shape[0]

# 2) Compute global P90 per (Mysql02, kpi_name) using full series (do NOT filter first)
# Reuse global_p90 if exists; otherwise compute
try:
    global_p90
except NameError:
    global_p90 = (
        df_metric
        .dropna(subset=['value'])
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.90)
        .rename(columns={'value': 'global_p90'})
    )

mysql_p90 = global_p90[global_p90['cmdb_id'] == mysql_cm].copy()

# 3) Filter Mysql02 rows to timestamps between 2021-03-25 02:55:00 and 2021-03-25 02:59:00 (UTC+8)
window_start_sh = pd.Timestamp('2021-03-25 02:55:00').tz_localize(sh)
window_end_sh = pd.Timestamp('2021-03-25 02:59:00').tz_localize(sh)
mysql_window = mysql_df_all[(mysql_df_all['ts_sh'] >= window_start_sh) & (mysql_df_all['ts_sh'] <= window_end_sh)].copy()

# 4) For each kpi compute required summary and samples
if mysql_window.empty:
    result_df = pd.DataFrame(columns=[
        'kpi_name','global_p90','points_in_window','count_above_p90',
        'earliest_timestamp_above_p90','max_value_in_window','sample_rows_recent_first'
    ])
else:
    merged = mysql_window.merge(mysql_p90, on=['cmdb_id','kpi_name'], how='left')
    # mark strict > p90
    merged['above_p90'] = merged['value'] > merged['global_p90']
    # aggregate per kpi
    agg = merged.groupby('kpi_name', as_index=False).agg(
        points_in_window=('value','size'),
        count_above_p90=('above_p90','sum'),
        max_value_in_window=('value','max')
    )
    # earliest timestamp above p90
    earliest = (
        merged[merged['above_p90']]
        .groupby('kpi_name', as_index=False)['ts_sh']
        .min()
        .rename(columns={'ts_sh':'earliest_ts_above_p90'})
    )
    agg = agg.merge(earliest, on='kpi_name', how='left')
    agg = agg.merge(mysql_p90[['kpi_name','global_p90']], on='kpi_name', how='left')
    # samples: up to 10 most recent rows per kpi (most recent first)
    samples = {}
    grp = merged.sort_values('timestamp', ascending=False).groupby('kpi_name')
    for kpi, g in grp:
        rows = g[['ts_sh','value','above_p90']].head(10).copy()
        rows['ts_str'] = rows['ts_sh'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        # build compact tuples (timestamp_str, value, above_p90)
        samples[kpi] = list(rows[['ts_str','value','above_p90']].itertuples(index=False, name=None))
    agg['sample_rows_recent_first'] = agg['kpi_name'].apply(lambda k: samples.get(k, []))
    # format earliest timestamp strings
    agg['earliest_timestamp_above_p90'] = agg['earliest_ts_above_p90'].apply(lambda x: x.strftime('%Y-%m-%d %H:%M:%S %Z') if pd.notna(x) else None)
    result_df = agg[[
        'kpi_name','global_p90','points_in_window','count_above_p90',
        'earliest_timestamp_above_p90','max_value_in_window','sample_rows_recent_first'
    ]].copy()

# 5) Sort by earliest_timestamp_above_p90 ascending (nulls last), limit to 20 rows
# For sorting, convert earliest_timestamp_above_p90 back to timestamp, with NaT for None
def to_ts(x):
    if pd.isna(x) or x is None:
        return pd.NaT
    return pd.to_datetime(x).tz_localize(sh)

result_df['earliest_ts_sort'] = pd.to_datetime(result_df['earliest_timestamp_above_p90']).dt.tz_localize(sh, ambiguous='NaT', nonexistent='NaT')
# Place NaT last by using fillna with large timestamp
large_ts = pd.Timestamp('2099-01-01').tz_localize(sh)
result_df['earliest_sort_key'] = result_df['earliest_ts_sort'].fillna(large_ts)
result_df_sorted = result_df.sort_values(['earliest_sort_key','count_above_p90','max_value_in_window'], ascending=[True, False, False]).head(20)

# Keep output compact: select columns to display
output_df = result_df_sorted[[
    'kpi_name','global_p90','points_in_window','count_above_p90',
    'earliest_timestamp_above_p90','max_value_in_window','sample_rows_recent_first'
]].reset_index(drop=True)

# Also provide small metadata: number of kpis and rows in window
meta = pd.DataFrame({
    'mysql_cmdb_id':[mysql_cm],
    'distinct_kpi_count':[mysql_kpi_count],
    'rows_in_window':[int(mysql_window.shape[0])],
    'kpis_with_any_above_p90':[int((result_df['count_above_p90']>0).sum()) if not result_df.empty else 0]
})

# Display compact results (DataFrame variables)
mysql_kpi_df, meta, output_df

```
Out[10]:
```
```python
# Plain-English summary of Mysql02 findings (from metric.csv analysis)
summary = (
    "Mysql02 DB-side timing summary (window 2021-03-25 02:55:00–02:59:00 UTC+8):\n\n"
    "- Scope: Mysql02 has 224 distinct KPIs; 820 metric rows were present in the 02:55–02:59 window. "
    "54 different Mysql02 KPIs had at least one value > their global P90 in that window.\n\n"
    "- Earliest breach: The first KPI to exceed its global P90 was 'Mysql-MySQL_3306_Bytes Received' at 02:55:00 (two points > P90 around 02:55–02:59).\n\n"
    "- Strong DB signals at ~02:57: Several important DB KPIs breached P90 at ~02:57:00, notably:\n"
    "  * Mysql-MySQL_3306_Sort Rows — 5 points in-window, 3 > P90, earliest >P90 at 02:57:00, max ≈ 1,213,281. (Indicates heavy sorting activity.)\n"
    "  * Multiple InnoDB-related metrics (data written, buffer pool read/write requests, os log written) show >P90 at 02:57 with one or more points above threshold.\n"
    "  * Query/Questions/Rows Read metrics also show elevated values at 02:57.\n\n"
    "- Timing relationship: DB anomalies concentrate at 02:57, immediately before the Tomcat request/processing spikes observed around 02:58. This suggests DB-side load or contention beginning ~02:57 that could contribute to downstream latency observed in Tomcat at ~02:58.\n\n"
    "- Recommended next steps: investigate DB activity at 02:56–02:58 (slow queries, long sorts, large writes, fsyncs), correlate slow query logs and InnoDB metrics with Tomcat request spikes, and check for temporary resource contention (I/O or locks) causing elevated Sort Rows and InnoDB operations.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name
0                    Mysql-MySQL_3306_Aborted Clients
1                   Mysql-MySQL_3306_Aborted Connects
2              Mysql-MySQL_3306_Binlog Cache Disk Use
3                   Mysql-MySQL_3306_Binlog Cache Use
4         Mysql-MySQL_3306_Binlog stmt cache disk use
..                                                ...
219                      OSLinux-OSLinux_SWAP_SWAP_Si
220                      OSLinux-OSLinux_SWAP_SWAP_So
221  OSLinux-OSLinux_SYSTEM_SYSTEM_Check-DefaultRoute
222      OSLinux-OSLinux_SYSTEM_SYSTEM_Check-Hostname
223                OSLinux-OSLinux_ZABBIX_Host_Uptime

[224 rows x 1 columns],   mysql_cmdb_id  distinct_kpi_count  rows_in_window  kpis_with_any_above_p90
0       Mysql02                 224             820                       54,                                              kpi_name    global_p90  points_in_window  count_above_p90 earliest_timestamp_above_p90  max_value_in_window                           sample_rows_recent_first
0                     Mysql-MySQL_3306_Bytes Received -5.928167e+02                 4                2      2021-03-25 02:55:00 CST        -1.588333e+02  [(2021-03-25 02:59:00 CST, -2869.2787, False),...
1                     OSLinux-CPU_CPU-3_SingleCpuidle  9.920048e+01                 2                1      2021-03-25 02:56:00 CST         9.938600e+01  [(2021-03-25 02:58:00 CST, 98.5731, False), (2...
2                     OSLinux-CPU_CPU-0_SingleCpuidle  9.901798e+01                 2                1      2021-03-25 02:56:00 CST         9.933670e+01  [(2021-03-25 02:58:00 CST, 98.6572, False), (2...
3                     OSLinux-CPU_CPU-1_SingleCpuidle  9.926388e+01                 2                1      2021-03-25 02:56:00 CST         9.930340e+01  [(2021-03-25 02:58:00 CST, 99.254, False), (20...
4                         OSLinux-CPU_CPU_CPUidleutil  9.905482e+01                 2                1      2021-03-25 02:56:00 CST         9.920800e+01  [(2021-03-25 02:59:00 CST, 97.7453, False), (2...
5    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  4.213300e+00                 2                1      2021-03-25 02:56:00 CST         4.533300e+00  [(2021-03-25 02:58:00 CST, 2.8, False), (2021-...
6     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps  3.366400e-01                 2                1      2021-03-25 02:56:00 CST         3.500000e-01  [(2021-03-25 02:58:00 CST, 0.25, False), (2021...
7                          Mysql-MySQL_3306_Sort Rows  1.213217e+06                 5                3      2021-03-25 02:57:00 CST         1.213281e+06  [(2021-03-25 02:59:00 CST, 1213281.0, True), (...
8                Mysql-MySQL_3306_Innodb data written  2.816488e+12                 5                1      2021-03-25 02:57:00 CST         2.816568e+12  [(2021-03-25 02:59:00 CST, 283681.5738, False)...
9   Mysql-MySQL_3306_Innodb buffer pool read requests  1.665579e+12                 5                1      2021-03-25 02:57:00 CST         1.665580e+12  [(2021-03-25 02:59:00 CST, 69.4262, False), (2...
10             Mysql-MySQL_3306_Innodb os log written  7.410132e+11                 5                1      2021-03-25 02:57:00 CST         7.410148e+11  [(2021-03-25 02:59:00 CST, 6093.6393, False), ...
11                         Mysql-MySQL_3306_Rows Read  5.444790e+11                 5                1      2021-03-25 02:57:00 CST         5.444790e+11  [(2021-03-25 02:59:00 CST, 6.5902, False), (20...
12                 Mysql-MySQL_3306_Handler Read Prev  5.442880e+11                 5                1      2021-03-25 02:57:00 CST         5.442880e+11  [(2021-03-25 02:59:00 CST, 1.5246, False), (20...
13                        Mysql-MySQL_3306_Bytes Sent  1.898770e+11                 5                1      2021-03-25 02:57:00 CST         1.898777e+11  [(2021-03-25 02:59:00 CST, 2758.7869, False), ...
14  Mysql-MySQL_3306_Innodb buffer pool write requ...  3.148634e+09                 5                1      2021-03-25 02:57:00 CST         3.148643e+09  [(2021-03-25 02:59:00 CST, 32.0, False), (2021...
15                    Mysql-MySQL_3306_MySQL  Queries  1.541196e+09                 5                1      2021-03-25 02:57:00 CST         1.541199e+09  [(2021-03-25 02:59:00 CST, 12.918, False), (20...
16                         Mysql-MySQL_3306_Questions  9.860242e+08                 5                1      2021-03-25 02:57:00 CST         9.860261e+08  [(2021-03-25 02:59:00 CST, 8.2295, False), (20...
17         Mysql-MySQL_3306_Innodb log write requests  8.922449e+08                 5                1      2021-03-25 02:57:00 CST         8.922465e+08  [(2021-03-25 02:59:00 CST, 6.4426, False), (20...
18                    Mysql-MySQL_3306_Handler Commit  8.213254e+08                 5                1      2021-03-25 02:57:00 CST         8.213269e+08  [(2021-03-25 02:59:00 CST, 7.459, False), (202...
19                        Mysql-MySQL_3306_Com Select  5.608799e+08                 5                1      2021-03-25 02:57:00 CST         5.608809e+08  [(2021-03-25 02:59:00 CST, 4.8852, False), (20...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```